# รายงานผลการดำเนินงาน Sprint 3 (Final Term Project)

## ระบบค้นพบภาพยนตร์และสร้างรายการรับชม (Movie Discovery & Watchlist Builder)

**รายวิชา:** CP352301 Script Programming (1/2569)
**Sprint 3:** Full-Stack App Dev — Web App (สัปดาห์ที่ 14)
**กำหนดนำเสนอ:** 29–30/9/69
**กำหนดส่งงาน:** 2/10/69

| บทบาท (หมุนเวียนตาม Sprint) | สมาชิกในทีม |
|---|---|
| Planner / Team Leader | นางสาวพลอยชมพู วงศ์กีรติกุล |
| Coder | นายอเสข ปัญญาวงค์ |
| Debugger / QA | นายนวมินทร์ คำจันทร์ |

**Repository:** https://github.com/somjung/Movie-Discovery

---

เนื้อหาของรายงานแบ่งออกเป็น 4 ส่วน ดังนี้

1. **ส่วนที่ 1: บทสรุปโครงการ (Project Recap)** ได้แก่ ภาพรวมของระบบ สถาปัตยกรรม และสิ่งที่ Sprint 3 เพิ่มจากรอบก่อน
2. **ส่วนที่ 2: แผนงาน (Plan)** ได้แก่ เป้าหมาย ขอบเขต หน้าจอ/เส้นทาง และกรณีขอบเขต
3. **ส่วนที่ 3: การพัฒนา (Execution)** ได้แก่ โค้ดที่ส่งมอบทั้งชุด และสาธิตการทำงานจริงแบบออฟไลน์
4. **ส่วนที่ 4: ผลลัพธ์และการทดสอบ (Result)** ได้แก่ สรุปความก้าวหน้า ตาราง QA บทเรียน และลิงก์ของโปรเจกต์


## ส่วนที่ 1: บทสรุปโครงการ (Project Recap)

### 1. ภาพรวมระบบ

ระบบค้นพบภาพยนตร์และสร้างรายการรับชม (Movie Discovery & Watchlist Builder)
เป็นแอปพลิเคชันที่เปลี่ยน "ภาพยนตร์ที่ผู้ใช้ชื่นชอบ" ให้กลายเป็น "รายการรับชม" ที่รวบรวมไว้ในที่เดียว
ผู้ใช้ระบุภาพยนตร์ตั้งต้น (Seed Movie) หรือคำค้น ระบบดึงข้อมูลจริงจาก TMDB API (The Movie Database)
กรองและจัดอันดับตามเงื่อนไขที่ต้องการ บันทึกผลลงฐานข้อมูล SQLite ส่วนตัว แล้วส่งออกเป็นไฟล์ CSV

Sprint 3 เป็นรอบ Full-Stack ตามคู่มือรายวิชา: เชื่อมชั้นนำเสนอเข้ากับระบบหลังบ้านให้สมบูรณ์
จัดการสถานะ (State) และความสอดคล้องของข้อมูลทั่วระบบ และรับมือกรณีขอบเขตครบถ้วน —
รอบนี้เพิ่ม **เว็บแอปพลิเคชัน** เป็นชั้นนำเสนอใหม่ (คู่มืออนุญาต CLI หรือ GUI)

### 2. สถาปัตยกรรม 3 ชั้นหลังรอบนี้

โครง 3 ชั้นของ Sprint 1–2 คงอยู่ทั้งหมด — Sprint 3 เปลี่ยนเฉพาะชั้นนำเสนอ:

```
        ┌──────────────────────────────────────────────┐
        │  Presentation Layer                          │
        │  web/ (Flask)  ·  cli.py · app.py            │
        └───────────────────────┬──────────────────────┘
                                │  ใช้บริการชุดเดียวกัน
        ┌───────────────────────▼──────────────────────┐
        │  Business Logic Layer                        │
        │  discovery.py · watchlist.py                 │
        └───────────────────────┬──────────────────────┘
                                │  constructor injection
        ┌───────────────────────▼──────────────────────┐
        │  Data Access Layer                           │
        │  tmdb_client.py · movie_store.py · exporter.py │
        └──────────────────────────────────────────────┘
              │                  │               │
        TMDB REST API     data/movies.db    data/*.csv
```
*(ภาพที่ 1: โครงสร้าง 3 ชั้นและทิศทางการพึ่งพาของระบบหลัง Sprint 3)*

- เว็บนำเข้า (import) โมดูลเดิมจาก `src/` โดยไม่คัดลอกตรรกะซ้ำ — และไม่แก้โค้ด `src/` ที่ส่งมอบแล้ว
- เว็บกับ CLI ใช้บริการและฐานข้อมูลชุดเดียวกัน (`data/movies.db`) — เพิ่มรายการจากฝั่งหนึ่ง อีกฝั่งเห็นทันที
- การเชื่อมต่อ SQLite เปิด–ปิดใหม่ต่อหนึ่งคำขอ เพราะเซิร์ฟเวอร์เว็บทำงานหลาย thread

### 3. สิ่งที่ Sprint 3 เพิ่มจากรอบก่อน

- เว็บแอปพลิเคชัน (Flask) ครบทุกหน้าจอ: หน้าแรก · ค้นหา · รายละเอียดหนัง + หนังคล้ายพร้อมตัวกรอง · รายการรับชม · รายการโปรด (ป้าย ★ ไขว้ลิสต์) · ประวัติ · ดาวน์โหลด CSV · หน้าข้อผิดพลาดแบบสุภาพ (404 · TMDB ล่ม 502 · ฐานข้อมูลเสีย)
- การจัดการ State และความสอดคล้องของข้อมูล: จำนวนรายการบนแถบนำทางมาจากฐานข้อมูลจริง อัปเดตทันทีหลังทุกการเพิ่ม/ลบ · ไฟล์ CSV ตรงกับลิสต์ปัจจุบันเสมอ
- โค้ด src/ ทั้ง 10 โมดูลเหมือนรอบ Sprint 2 ทุกไบต์ (ตรวจเทียบแล้ว) — เพิ่มเฉพาะ web/ และชุดทดสอบเว็บ
- ชุดทดสอบ 133 เคส (เดิม 93 + ใหม่ 40 สำหรับเว็บ) ออฟไลน์ 100% · flake8 ผ่านทั้งโปรเจกต์รวม web/


## ส่วนที่ 2: แผนงาน (Plan)

### 2.1 เป้าหมายของ Sprint 3

คู่มือรายวิชากำหนด Sprint 3 เป็น Full-Stack App Dev: เชื่อมต่อ Front-End และ Back-End ให้สมบูรณ์ ·
จัดการ State · ความสอดคล้องของข้อมูล · การรับมือกรณีขอบเขต (Edge Cases) รอบนี้จึง:

1. สร้างส่วนปฏิสัมพันธ์กับผู้ใช้ใหม่เป็นเว็บแอปพลิเคชัน เชื่อมกับ Business Logic และ Data Access เดิมทั้งหมด
2. จัดการ State และความสอดคล้องของข้อมูลในมุมมองเว็บ (จำนวนรายการ · ป้ายไขว้ลิสต์ · ข้อมูลชุดเดียวกับ CLI)
3. รับมือ Edge Cases ครบทั้งระบบ: คีย์หาย · เน็ตหลุด · ฐานข้อมูลเสีย · URL ผิด
4. ต่อยอดชุดทดสอบอัตโนมัติ (Flask test client + ตัวจำลองเดิม) และคง CI ให้เขียวต่อเนื่อง

### 2.2 ขอบเขต

- **ในขอบเขต:** เว็บครบทุกหน้าจอหลัก · ปุ่มเพิ่ม/ลบ/เคลียร์จากหน้าเว็บ · ดาวน์โหลด CSV · ป้าย ★ · กรณีขอบเขตตามหัวข้อ 2.4 · เทสต์ใหม่ + CI
- **นอกขอบเขต:** การพัฒนา CLI เพิ่มเติม (หยุดไว้ที่ Sprint 2 — โค้ดคงอยู่ ไม่ลบ) · ฟีเจอร์ค้นหาเชิงลึก (ประเภท/นักแสดง · ช่องทางรับชม) ยกไป Sprint สุดท้าย · การติดตั้งบนคลาวด์ · ระบบบัญชีผู้ใช้
- **ข้อกำหนด:** คีย์ API เก็บผ่านตัวแปรสภาพแวดล้อม `TMDB_API_KEY` เท่านั้น

### 2.3 หน้าจอและเส้นทาง (Route Specification)

| เส้นทาง | หน้าที่ |
|---|---|
| `GET /` | หน้าแรก — ช่องค้นหา · ลิงก์ทุกหน้า · จำนวนรายการบนแถบนำทาง · แบนเนอร์เมื่อไม่มีคีย์ |
| `GET /search?q=` | ผลการค้นหา (การ์ดหนัง) · บันทึกประวัติอัตโนมัติ |
| `GET /movie/<id>` | รายละเอียดหนัง + หนังคล้ายพร้อมตัวกรอง (คะแนน · โหวต · ปี · เรียง · จำนวน) + ปุ่มเพิ่มสองลิสต์ |
| `POST /watchlist/add/<id>` · `/remove/<id>` · `/clear` | จัดการรายการรับชมจากหน้าเว็บ |
| `POST /favorites/add/<id>` · `/remove/<id>` | จัดการรายการโปรดจากหน้าเว็บ |
| `GET /watchlist` · `GET /favorites` | แสดงลิสต์ + ป้าย ★ + ดาวน์โหลด CSV + ลบรายรายการ |
| `GET /export/<kind>` | ดาวน์โหลดไฟล์ CSV (watchlist / favorites) |
| `GET /history` | ประวัติการค้นหาล่าสุด |
| เส้นทางอื่น / ข้อผิดพลาด | 404 · TMDB ล่ม 502 · ฐานข้อมูลเสีย → หน้าข้อผิดพลาดแบบสุภาพ |

### 2.4 กรณีขอบเขต (Edge Cases)

| กรณี | พฤติกรรมที่ต้องเกิด |
|---|---|
| ไม่มีคีย์ TMDB | ทุกหน้าเปิดได้ · หน้าค้นหาแจ้งวิธีตั้งค่าคีย์ · หน้าออฟไลน์ (ลิสต์/ประวัติ/CSV) ใช้ได้ปกติ |
| TMDB ล่ม / timeout | หน้าแจ้งแบบสุภาพ + ลองใหม่ได้ · ถ้าแคชยังสดแสดงผลจากแคชได้ |
| คีย์ผิด (401) · เรียกถี่เกิน (429) | ข้อความอธิบายชัดเจนตามแบบ Sprint 2 |
| คำค้นว่าง / ยาวเกิน / ไทย / emoji | ค้นได้หรือแจ้ง "ไม่พบ" — ไม่พัง |
| รหัสหนังไม่มีจริง (404) | หน้า "ไม่พบภาพยนตร์" แบบสุภาพ |
| ไฟล์ฐานข้อมูลเสีย | หน้าแจ้งปัญหา + วิธีแก้ (ย้าย/ลบไฟล์แล้วเริ่มใหม่) |
| เพิ่มซ้ำ · ลบของที่ไม่มี · URL ไม่มีอยู่ | ข้อความแจ้งผลชัดเจน · 404 แบบสุภาพ |

### 2.5 Definition of Done (สรุป — ผลการตรวจอยู่ในส่วนที่ 4)

- เปิดเว็บได้ครบทุกหน้า · ค้นหา/กรองเสมือน CLI · ลิสต์ + ★ + CSV ตรงกับข้อมูลจริง
- CLI เดิมยังทำงาน · ข้อมูลอยู่ข้ามการรีสตาร์ท
- flake8 + pytest ผ่านครบ · ไม่มีคีย์ในโค้ด/เทมเพลต/commit · CI เขียวต่อเนื่อง


## ส่วนที่ 3: การพัฒนา (Execution)

ลำดับการพัฒนา: ① วางโครง `web/` และเพิ่ม dependency → ② สร้างหน้าจอและเส้นทางทั้งหมด →
③ Hardening ตามตารางกรณีขอบเขต → ④ เขียนชุดทดสอบเว็บใหม่ 40 เคส →
⑤ ตรวจรวม 133 เคส + flake8 ทั้งโปรเจกต์

โค้ดที่ส่งมอบทั้งชุดถูกฝังในเซลล์ด้านล่าง **ตามไฟล์จริงในโครงการทุกไบต์** —
โน้ตบุ๊กนี้สร้างด้วยสคริปต์ที่อ่านไฟล์จากโฟลเดอร์โค้ดโดยตรง รายงานกับโค้ดจึงไม่คลาดกัน
จากนั้นเซลล์สาธิตทั้งหมดจะรันจริงแบบ **ออฟไลน์ 100%** ด้วยตัวจำลอง TMDB ชุดเดียวกับชุดทดสอบ
(ไม่ต้องใช้คีย์หรืออินเทอร์เน็ต — รันซ้ำได้ทุกเครื่องและบน CI)



In [1]:
# เตรียมโครงสร้างโปรเจกต์และย้ายเข้าไปทำงานในโฟลเดอร์
import os
import pathlib
import sys

here = pathlib.Path.cwd()
if (here / "src" / "cli.py").exists():
    ROOT = here  # อยู่ในโฟลเดอร์โปรเจกต์อยู่แล้ว
else:
    ROOT = here / "script-final-project"
    if (ROOT / "src" / "cli.py").exists():
        # รันข้างโฟลเดอร์โปรเจกต์เดิม: ใช้โฟลเดอร์ใหม่ ไม่เขียนทับของเดิม
        ROOT = here / "script-final-project-notebook"
for folder in ("src", "tests", "data", "web", "web/templates", "web/static"):
    (ROOT / folder).mkdir(parents=True, exist_ok=True)
os.chdir(ROOT)
ROOT = pathlib.Path.cwd()
for path in (ROOT, ROOT / "tests"):
    if str(path) not in sys.path:
        sys.path.insert(0, str(path))
print("working directory:", ROOT)
print("python:", sys.version.split()[0])


working directory: C:\Users\User\AppData\Local\hermes\profiles\generalist\cache\scratch\nb_run\s3_v3\script-final-project
python: 3.12.13


In [2]:
# ติดตั้งไลบรารีที่ใช้รันและทดสอบ (ถ้ามีอยู่แล้วระบบจะข้ามให้)
%pip install -q pytest flake8 requests flask
import flask
import pytest
print("pytest", pytest.__version__, "| flask", flask.__version__)


Note: you may need to restart the kernel to use updated packages.


pytest 9.1.1 | flask 3.1.3


C:\Users\User\AppData\Local\Temp\ipykernel_19188\4002804723.py:5: DeprecationWarning: The '__version__' attribute is deprecated and will be removed in Flask 3.2. Use feature detection or 'importlib.metadata.version("flask")' instead.
  print("pytest", pytest.__version__, "| flask", flask.__version__)


In [3]:
%%writefile requirements.txt
requests>=2.31
flask>=3.0
pytest>=8.0
flake8>=7.0


Writing requirements.txt


In [4]:
%%writefile setup.cfg
[flake8]
max-line-length = 99
exclude = .venv,__pycache__,.git,.pytest_cache

[tool:pytest]
testpaths = tests


Writing setup.cfg


In [5]:
%%writefile .env.example
# TMDB API key (free for non-commercial use, attribution required).
# Create one at https://developer.themoviedb.org/ and paste it below,
# then copy this file to .env (the .env file is never committed).
TMDB_API_KEY=your_tmdb_api_key_here


Writing .env.example


In [6]:
%%writefile src/__init__.py
"""Movie Discovery & Watchlist Builder — core package."""

__version__ = "0.1.0"


Writing src/__init__.py


In [7]:
%%writefile src/config.py
"""Configuration constants and API-key lookup for the project."""

import os

TMDB_BASE_URL = "https://api.themoviedb.org/3"
DEFAULT_LANGUAGE = "en-US"
DEFAULT_TIMEOUT = 10

PROJECT_ROOT = os.path.dirname(os.path.dirname(os.path.abspath(__file__)))


def get_data_dir():
    """Folder for runtime data: the SQLite database and the CSV exports."""
    return os.path.join(PROJECT_ROOT, "data")


def get_db_path():
    """Path of the project's SQLite file (``data/movies.db`` under the root)."""
    return os.path.join(get_data_dir(), "movies.db")


def get_api_key():
    """Return the TMDB API key from the environment.

    Raises RuntimeError with a helpful message when the key is missing.
    """
    api_key = os.environ.get("TMDB_API_KEY", "").strip()
    if not api_key:
        raise RuntimeError(
            "TMDB_API_KEY is not set. Create a free key at "
            "https://developer.themoviedb.org/ and export it, e.g. "
            "set TMDB_API_KEY=your_key_here"
        )
    return api_key


Writing src/config.py


In [8]:
%%writefile src/sample_data.py
"""Built-in sample dataset used by the Sprint 1 CLI and the test-suite.

The TMDB-backed data layer arrives in Sprint 2; until then the CLI is
demonstrated safely offline with these five movies.
"""

SAMPLE_MOVIES = [
    {"id": 101, "title": "Alpha Signal", "release_date": "2010-05-01",
     "vote_average": 7.8, "vote_count": 1200, "popularity": 40.0},
    {"id": 102, "title": "Bravo Horizon", "release_date": "2014-09-12",
     "vote_average": 6.1, "vote_count": 300, "popularity": 22.0},
    {"id": 103, "title": "Cobalt Night", "release_date": "1999-01-20",
     "vote_average": 8.4, "vote_count": 2100, "popularity": 55.0},
    {"id": 104, "title": "Delta Echo", "release_date": "",
     "vote_average": 5.0, "vote_count": 50, "popularity": 5.0},
    {"id": 105, "title": "Emerald Run", "release_date": "2021-11-05",
     "vote_average": 8.0, "vote_count": 900, "popularity": 30.0},
]


Writing src/sample_data.py


In [9]:
%%writefile src/tmdb_client.py
"""TMDB API client: read-only GET access with explicit error handling and an
optional SQLite response cache.
"""

import json
from urllib.parse import urlencode

import requests

from .config import TMDB_BASE_URL, DEFAULT_LANGUAGE, DEFAULT_TIMEOUT, get_api_key
from .movie_store import CACHE_TTL_SECONDS


class TmdbError(Exception):
    """Raised when a TMDB request cannot be completed."""


class TmdbClient:
    """Minimal TMDB REST client (only the endpoints this project needs).

    Pass a ``store`` to enable response caching in its ``api_cache`` table:
    a request that is still fresh is served from SQLite without any network
    call; everything else is fetched and then stored for next time.
    """

    def __init__(self, api_key=None, base_url=TMDB_BASE_URL,
                 timeout=DEFAULT_TIMEOUT, session=None, store=None,
                 cache_ttl=CACHE_TTL_SECONDS):
        """Build a client; ``store=None`` disables the response cache."""
        self.api_key = api_key if api_key is not None else get_api_key()
        self.base_url = base_url.rstrip("/")
        self.timeout = timeout
        self.session = session if session is not None else requests.Session()
        self.store = store
        self.cache_ttl = cache_ttl
        # Counters the CLI and tests can read to show the cache working.
        self.cache_hits = 0
        self.cache_misses = 0

    def _cache_key(self, path, query):
        """Deterministic key: the path plus every parameter except the API key."""
        public = sorted((key, value) for key, value in query.items()
                        if key != "api_key")
        return f"{path}?{urlencode(public)}"

    def _read_cache(self, cache_key):
        """Fresh cached body as a dict, or None (absent, expired, or corrupt)."""
        if self.store is None:
            return None
        cached = self.store.get_cached(cache_key, self.cache_ttl)
        if cached is None:
            return None
        try:
            return json.loads(cached)
        except ValueError:
            return None  # corrupt entry: ignore it and refetch from the API

    def _write_cache(self, cache_key, payload):
        """Store a fresh response body for future reads (no-op without a store)."""
        if self.store is not None:
            self.store.save_cached(cache_key, json.dumps(payload))

    def _get(self, path, params=None):
        """GET ``path`` and return the decoded JSON body.

        Served from the cache when fresh; otherwise fetched from the network
        and stored. Every failure mode raises TmdbError with a readable
        message: network problems, auth failures, missing resources, rate
        limits, unexpected statuses, and invalid JSON.
        """
        query = {"api_key": self.api_key, "language": DEFAULT_LANGUAGE}
        if params:
            query.update(params)
        cache_key = self._cache_key(path, query)

        cached = self._read_cache(cache_key)
        if cached is not None:
            self.cache_hits += 1
            return cached
        if self.store is not None:
            self.cache_misses += 1

        try:
            response = self.session.get(
                f"{self.base_url}{path}", params=query, timeout=self.timeout
            )
        except requests.RequestException as exc:
            raise TmdbError(f"request to {path} failed: {exc}") from exc

        if response.status_code == 401:
            raise TmdbError("TMDB rejected the API key (HTTP 401)")
        if response.status_code == 404:
            raise TmdbError(f"TMDB resource not found (HTTP 404): {path}")
        if response.status_code == 429:
            raise TmdbError("TMDB rate limit hit (HTTP 429) - slow down")
        if response.status_code != 200:
            raise TmdbError(
                f"unexpected TMDB status {response.status_code} for {path}"
            )
        try:
            payload = response.json()
        except ValueError as exc:
            raise TmdbError(f"TMDB returned invalid JSON for {path}") from exc
        self._write_cache(cache_key, payload)
        return payload

    def search_movies(self, query, page=1):
        """Search movies by title. Returns a list of movie dicts."""
        data = self._get("/search/movie", {"query": query, "page": page})
        return data.get("results", [])

    def get_movie_details(self, tmdb_id):
        """Full details for one movie id (used when adding it by id)."""
        return self._get(f"/movie/{tmdb_id}")

    def get_similar(self, tmdb_id, page=1):
        """Movies similar to a given movie id."""
        data = self._get(f"/movie/{tmdb_id}/similar", {"page": page})
        return data.get("results", [])

    def get_recommendations(self, tmdb_id, page=1):
        """TMDB's recommendations for a given movie id."""
        data = self._get(f"/movie/{tmdb_id}/recommendations", {"page": page})
        return data.get("results", [])


Writing src/tmdb_client.py


In [10]:
%%writefile src/movie_store.py
"""SQLite persistence layer: movies, similar-links, watchlist, favorites,
search history, and the TMDB response cache.
"""

import os
import sqlite3

# How long a cached TMDB response stays fresh (24 hours, in seconds).
CACHE_TTL_SECONDS = 24 * 60 * 60

SCHEMA = """
CREATE TABLE IF NOT EXISTS movies (
    tmdb_id      INTEGER PRIMARY KEY,
    title        TEXT NOT NULL,
    release_date TEXT,
    vote_average REAL,
    vote_count   INTEGER,
    popularity   REAL,
    overview     TEXT,
    added_at     TEXT NOT NULL DEFAULT (datetime('now'))
);

CREATE TABLE IF NOT EXISTS similar_links (
    source_id INTEGER NOT NULL,
    target_id INTEGER NOT NULL,
    score     REAL,
    PRIMARY KEY (source_id, target_id)
);

CREATE TABLE IF NOT EXISTS watchlist (
    movie_id INTEGER PRIMARY KEY,
    note     TEXT,
    added_at TEXT NOT NULL DEFAULT (datetime('now'))
);

CREATE TABLE IF NOT EXISTS favorites (
    movie_id INTEGER PRIMARY KEY,
    note     TEXT,
    added_at TEXT NOT NULL DEFAULT (datetime('now'))
);

CREATE TABLE IF NOT EXISTS searches (
    id           INTEGER PRIMARY KEY AUTOINCREMENT,
    query        TEXT NOT NULL,
    result_count INTEGER NOT NULL DEFAULT 0,
    searched_at  TEXT NOT NULL DEFAULT (datetime('now'))
);

CREATE TABLE IF NOT EXISTS api_cache (
    cache_key  TEXT PRIMARY KEY,
    payload    TEXT NOT NULL,
    fetched_at TEXT NOT NULL DEFAULT (datetime('now'))
);
"""


class MovieStore:
    """Small data-access layer on top of SQLite."""

    def __init__(self, db_path=":memory:"):
        """Open a connection and create the tables when they do not exist yet."""
        if db_path != ":memory:":
            parent = os.path.dirname(os.path.abspath(db_path))
            os.makedirs(parent, exist_ok=True)
        self.conn = sqlite3.connect(db_path)
        self.conn.row_factory = sqlite3.Row
        self.init_db()

    def close(self):
        """Close the underlying database connection."""
        self.conn.close()

    def init_db(self):
        """Create the tables when they do not exist yet."""
        with self.conn:
            self.conn.executescript(SCHEMA)

    def add_movie(self, movie):
        """Insert a movie (or refresh it when the id already exists)."""
        with self.conn:
            self.conn.execute(
                """
                INSERT INTO movies (tmdb_id, title, release_date, vote_average,
                                    vote_count, popularity, overview)
                VALUES (:tmdb_id, :title, :release_date, :vote_average,
                        :vote_count, :popularity, :overview)
                ON CONFLICT(tmdb_id) DO UPDATE SET
                    title = excluded.title,
                    release_date = excluded.release_date,
                    vote_average = excluded.vote_average,
                    vote_count = excluded.vote_count,
                    popularity = excluded.popularity,
                    overview = excluded.overview
                """,
                {
                    "tmdb_id": movie["id"],
                    "title": movie.get("title", ""),
                    "release_date": movie.get("release_date") or "",
                    "vote_average": movie.get("vote_average") or 0.0,
                    "vote_count": movie.get("vote_count") or 0,
                    "popularity": movie.get("popularity") or 0.0,
                    "overview": movie.get("overview") or "",
                },
            )
        return movie["id"]

    def get_movie(self, tmdb_id):
        """Return one movie as a dict, or None when it is not stored."""
        row = self.conn.execute(
            "SELECT * FROM movies WHERE tmdb_id = ?", (tmdb_id,)
        ).fetchone()
        return dict(row) if row else None

    def list_movies(self):
        """All stored movies (most recently added first)."""
        rows = self.conn.execute(
            "SELECT * FROM movies ORDER BY added_at DESC, tmdb_id"
        ).fetchall()
        return [dict(row) for row in rows]

    def add_similar_link(self, source_id, target_id, score=None):
        """Record that ``target_id`` is similar to ``source_id``."""
        with self.conn:
            self.conn.execute(
                """
                INSERT INTO similar_links (source_id, target_id, score)
                VALUES (?, ?, ?)
                ON CONFLICT(source_id, target_id) DO UPDATE SET
                    score = excluded.score
                """,
                (source_id, target_id, score),
            )

    def get_similar_links(self, source_id):
        """Similar-movie links recorded for a source movie (best first)."""
        rows = self.conn.execute(
            """
            SELECT target_id, score FROM similar_links
            WHERE source_id = ? ORDER BY score DESC
            """,
            (source_id,),
        ).fetchall()
        return [dict(row) for row in rows]

    def add_to_watchlist(self, movie_id, note=None):
        """Add a movie to the watchlist (no duplicates)."""
        with self.conn:
            self.conn.execute(
                """
                INSERT INTO watchlist (movie_id, note) VALUES (?, ?)
                ON CONFLICT(movie_id) DO UPDATE SET note = excluded.note
                """,
                (movie_id, note),
            )

    def is_in_watchlist(self, movie_id):
        """True when the movie id is already on the watchlist."""
        row = self.conn.execute(
            "SELECT 1 FROM watchlist WHERE movie_id = ?", (movie_id,)
        ).fetchone()
        return row is not None

    def get_watchlist(self):
        """Watchlist rows joined with movie details (newest first)."""
        rows = self.conn.execute(
            """
            SELECT w.movie_id AS tmdb_id, m.title, m.release_date,
                   m.vote_average, w.note, w.added_at
            FROM watchlist w
            LEFT JOIN movies m ON m.tmdb_id = w.movie_id
            ORDER BY w.added_at DESC, w.movie_id
            """
        ).fetchall()
        return [dict(row) for row in rows]

    def remove_from_watchlist(self, movie_id):
        """Delete one watchlist row; return True when a row was removed."""
        with self.conn:
            cur = self.conn.execute(
                "DELETE FROM watchlist WHERE movie_id = ?", (movie_id,)
            )
        return cur.rowcount > 0

    def clear_watchlist(self):
        """Delete every watchlist row; return how many entries were removed."""
        with self.conn:
            cur = self.conn.execute("DELETE FROM watchlist")
        return cur.rowcount

    def add_favorite(self, movie_id, note=None):
        """Add a movie to favorites (no duplicates; refreshes the note)."""
        with self.conn:
            self.conn.execute(
                """
                INSERT INTO favorites (movie_id, note) VALUES (?, ?)
                ON CONFLICT(movie_id) DO UPDATE SET note = excluded.note
                """,
                (movie_id, note),
            )

    def is_in_favorites(self, movie_id):
        """True when the movie id is already in favorites."""
        row = self.conn.execute(
            "SELECT 1 FROM favorites WHERE movie_id = ?", (movie_id,)
        ).fetchone()
        return row is not None

    def get_favorites(self):
        """Favorite rows joined with movie details (newest first)."""
        rows = self.conn.execute(
            """
            SELECT f.movie_id AS tmdb_id, m.title, m.release_date,
                   m.vote_average, f.note, f.added_at
            FROM favorites f
            LEFT JOIN movies m ON m.tmdb_id = f.movie_id
            ORDER BY f.added_at DESC, f.movie_id
            """
        ).fetchall()
        return [dict(row) for row in rows]

    def remove_favorite(self, movie_id):
        """Delete one favorites row; return True when a row was removed."""
        with self.conn:
            cur = self.conn.execute(
                "DELETE FROM favorites WHERE movie_id = ?", (movie_id,)
            )
        return cur.rowcount > 0

    def record_search(self, query, result_count):
        """Append one search to the history (query text + number of results)."""
        with self.conn:
            self.conn.execute(
                "INSERT INTO searches (query, result_count) VALUES (?, ?)",
                (query, result_count),
            )

    def get_recent_searches(self, limit=10):
        """Return the most recent searches first, up to ``limit`` rows."""
        rows = self.conn.execute(
            """
            SELECT id, query, result_count, searched_at FROM searches
            ORDER BY id DESC LIMIT ?
            """,
            (limit,),
        ).fetchall()
        return [dict(row) for row in rows]

    def get_cached(self, cache_key, ttl_seconds=CACHE_TTL_SECONDS):
        """Return a cached payload younger than the TTL, or None on a miss."""
        row = self.conn.execute(
            """
            SELECT payload FROM api_cache
            WHERE cache_key = ? AND fetched_at >= datetime('now', ?)
            """,
            (cache_key, f"-{max(0, int(ttl_seconds))} seconds"),
        ).fetchone()
        return row["payload"] if row else None

    def save_cached(self, cache_key, payload):
        """Store (or refresh) a response payload under ``cache_key``."""
        with self.conn:
            self.conn.execute(
                """
                INSERT INTO api_cache (cache_key, payload) VALUES (?, ?)
                ON CONFLICT(cache_key) DO UPDATE SET
                    payload = excluded.payload,
                    fetched_at = datetime('now')
                """,
                (cache_key, payload),
            )


Writing src/movie_store.py


In [11]:
%%writefile src/discovery.py
"""Discovery logic: search, filter, sort, dedupe and rank movie candidates."""

SORT_OPTIONS = ("rating", "votes", "year", "popularity")


class DiscoveryService:
    """Turns user queries and seed ids into ranked candidate lists.

    The API only supplies raw search/similar/recommended movies; which of
    them survive the filters and in which order is decided here (project
    logic), and everything shown to the user is kept in the local store.
    """

    def __init__(self, client, store):
        """Keep the API client and the local store the service writes to."""
        self.client = client
        self.store = store

    def search(self, query, limit=10):
        """Search TMDB for ``query``, log it in ``searches``, return top hits."""
        results = self.client.search_movies(query)
        self.store.record_search(query, len(results))
        return results[:limit]

    def find_similar(self, seed_id, min_rating=0.0, min_votes=0,
                     year_from=None, year_to=None, limit=10, sort="rating"):
        """Return the top ``limit`` candidates for the seed movie.

        Candidates come from the similar + recommended endpoints (merged,
        with duplicates removed), then are filtered, sorted by ``sort``
        and cut down to ``limit`` before being saved to the store.
        """
        candidates = {}
        for fetch in (self.client.get_similar, self.client.get_recommendations):
            for movie in fetch(seed_id):
                candidates.setdefault(movie["id"], movie)

        kept = [
            movie for movie in candidates.values()
            if self._passes_filters(movie, min_rating, min_votes,
                                    year_from, year_to)
        ]
        kept.sort(key=self._sort_key(sort), reverse=True)
        kept = kept[:limit]

        for movie in kept:
            self.store.add_movie(movie)
            self.store.add_similar_link(seed_id, movie["id"],
                                        movie.get("vote_average"))
        return kept

    @classmethod
    def _sort_key(cls, sort):
        """Pick the (primary, secondary) sort key; reject unknown names."""
        if sort not in SORT_OPTIONS:
            raise ValueError(f"unknown sort option: {sort!r}")
        keys = {
            "rating": lambda movie: (movie.get("vote_average") or 0,
                                     movie.get("vote_count") or 0),
            "votes": lambda movie: (movie.get("vote_count") or 0,
                                    movie.get("vote_average") or 0),
            "year": lambda movie: (cls.release_year(movie),
                                   movie.get("vote_average") or 0),
            "popularity": lambda movie: (movie.get("popularity") or 0,
                                         movie.get("vote_average") or 0),
        }
        return keys[sort]

    @staticmethod
    def release_year(movie):
        """Release year as int, or 0 when the date is unknown/invalid."""
        date = (movie.get("release_date") or "")[:4]
        return int(date) if date.isdigit() else 0

    @classmethod
    def _passes_filters(cls, movie, min_rating, min_votes, year_from, year_to):
        """True when the movie satisfies every active filter."""
        if (movie.get("vote_average") or 0) < min_rating:
            return False
        if (movie.get("vote_count") or 0) < min_votes:
            return False
        year = cls.release_year(movie)
        if year == 0 and (year_from or year_to):
            return False  # an unknown date cannot be placed inside a range
        if year_from and year < year_from:
            return False
        if year_to and year > year_to:
            return False
        return True


Writing src/discovery.py


In [12]:
%%writefile src/watchlist.py
"""Watchlist and favorites logic: Top-N building and single-movie adds."""


class WatchlistBuilder:
    """Builds the user's Top-N watchlist and adds movies by id.

    The optional ``client`` lets ``add``/``add_favorite`` fetch a movie
    from TMDB when it is not stored locally yet.
    """

    def __init__(self, store, client=None):
        """Keep the store and an optional TMDB client for fetch-by-id."""
        self.store = store
        self.client = client

    def build(self, candidates, top_n=10):
        """Pick up to ``top_n`` new movies and save them to the watchlist."""
        picked = []
        for movie in candidates:
            if self.store.is_in_watchlist(movie["id"]):
                continue
            picked.append(movie)
            if len(picked) >= top_n:
                break
        for movie in picked:
            self.store.add_to_watchlist(movie["id"])
        return picked

    def ensure_movie(self, movie_id):
        """Return the stored movie, fetching it from TMDB when it is missing."""
        movie = self.store.get_movie(movie_id)
        if movie is not None:
            return movie
        if self.client is None:
            raise ValueError(
                f"movie {movie_id} is not stored and no TMDB client is attached"
            )
        self.store.add_movie(self.client.get_movie_details(movie_id))
        return self.store.get_movie(movie_id)

    def add(self, movie_id, note=None):
        """Add one movie to the watchlist by id (fetching it when needed).

        Returns True when it was newly added, and False when it was
        already on the watchlist.
        """
        self.ensure_movie(movie_id)
        already = self.store.is_in_watchlist(movie_id)
        self.store.add_to_watchlist(movie_id, note)
        return not already

    def add_favorite(self, movie_id, note=None):
        """Add one movie to favorites by id (fetching it when needed).

        Returns True when it was newly added, and False when it was
        already in favorites.
        """
        self.ensure_movie(movie_id)
        already = self.store.is_in_favorites(movie_id)
        self.store.add_favorite(movie_id, note)
        return not already


Writing src/watchlist.py


In [13]:
%%writefile src/exporter.py
"""CSV export of the user's lists (watchlist and favorites)."""

import csv
import os

LIST_COLUMNS = ("tmdb_id", "title", "release_date", "vote_average", "note")
# Both lists share the same export schema; the old name stays for safety.
WATCHLIST_COLUMNS = LIST_COLUMNS


class ExportError(Exception):
    """Raised when a list cannot be written to disk."""


class Exporter:
    """Writes project data to CSV files."""

    @staticmethod
    def _write_rows(rows, path, columns):
        """Write ``rows`` to ``path``, creating the folder chain first.

        Raises ExportError (naming the failing path) when writing fails.
        """
        try:
            folder = os.path.dirname(os.path.abspath(path))
            os.makedirs(folder, exist_ok=True)
            with open(path, "w", newline="", encoding="utf-8") as handle:
                writer = csv.writer(handle)
                writer.writerow(columns)
                for row in rows:
                    writer.writerow([
                        "" if row.get(column) is None else row.get(column)
                        for column in columns
                    ])
        except OSError as exc:
            raise ExportError(f"cannot write {path}: {exc}") from exc

    @staticmethod
    def export_watchlist(store, path):
        """Write the current watchlist to ``path``; return how many rows."""
        rows = store.get_watchlist()
        Exporter._write_rows(rows, path, LIST_COLUMNS)
        return len(rows)

    @staticmethod
    def export_favorites(store, path):
        """Write the current favorites to ``path``; return how many rows."""
        rows = store.get_favorites()
        Exporter._write_rows(rows, path, LIST_COLUMNS)
        return len(rows)


Writing src/exporter.py


In [14]:
%%writefile src/cli.py
"""Interactive command-line front-end for the Movie Discovery app.

Sprint 2 scope: the presentation layer is now wired to the real back-end —
TMDB (with caching) for search/discover, SQLite for watchlist/favorites/
search history, and CSV export for the File I/O requirement.  Commands keep
the Sprint-1 Thai messages and validation; ``client`` and ``store`` can be
injected so the test-suite runs fully offline.
"""

import os

from .config import get_data_dir, get_db_path
from .discovery import SORT_OPTIONS, DiscoveryService
from .exporter import Exporter, ExportError
from .movie_store import MovieStore
from .tmdb_client import TmdbClient, TmdbError
from .watchlist import WatchlistBuilder

WELCOME_MESSAGE = """\
============================================================
  Movie Discovery & Watchlist Builder — CLI
  ระบบค้นพบภาพยนตร์และสร้างรายการรับชม
============================================================"""

MENU_TEXT = """\
คำสั่งที่ใช้ได้:
  search <ชื่อเรื่อง>              ค้นหาภาพยนตร์จาก TMDB
  discover <รหัส> [ตัวเลือก]       ภาพยนตร์ที่คล้ายกัน (กรอง/เรียงได้)
        --min-rating 0-10 · --min-votes N · --year YYYY|YYYY-YYYY
        --sort rating|votes|year|popularity · --limit N
  watchlist add <รหัส>            เพิ่มเข้ารายการรับชม
  watchlist list                  แสดงรายการรับชม
  watchlist remove <รหัส>         นำออกจากรายการรับชม
  watchlist clear                 ล้างรายการรับชมทั้งหมด
  favorites add <รหัส>            เพิ่มเข้ารายการโปรด
  favorites list                  แสดงรายการโปรด
  favorites remove <รหัส>         นำออกจากรายการโปรด
  export watchlist|favorites      ส่งออกรายการเป็น CSV (โฟลเดอร์ data/)
  history [จำนวน]                 ประวัติการค้นหาล่าสุด (ค่าเริ่มต้น 10)
  help                            แสดงคำสั่งทั้งหมด
  quit / exit / ออก               ออกจากโปรแกรม"""

FAREWELL_MESSAGE = "ขอบคุณที่ใช้งานโปรแกรม ลาก่อน"
QUIT_COMMANDS = {"quit", "exit", "q", "ออก"}

MISSING_KEY_MESSAGE = (
    "ยังไม่ได้ตั้งค่าคีย์ TMDB — กำหนดตัวแปรสภาพแวดล้อม TMDB_API_KEY "
    "ก่อนใช้งาน\n"
    "  · สร้างคีย์ฟรีได้ที่ https://developer.themoviedb.org/\n"
    "  · คำสั่งที่ไม่ใช้ TMDB (watchlist list, export, history, help) "
    "ยังใช้งานได้ตามปกติ"
)

DISCOVER_FLAGS = ("--min-rating", "--min-votes", "--year", "--sort", "--limit")

LIST_LABELS = {"watchlist": "รายการรับชม", "favorites": "รายการโปรด"}


class CLI:
    """Presentation layer of the project (menus + input validation)."""

    def __init__(self, input_func=input, output_func=print, client=None,
                 store=None, db_path=None, data_dir=None):
        """Wire the CLI; inject ``client``/``store`` to run offline (tests).

        Without a ``client`` the real TMDB client is built from
        ``TMDB_API_KEY``; when the key is missing the CLI still starts and
        only the TMDB-backed commands become unavailable.
        """
        self.input_func = input_func
        self.output_func = output_func
        self.store = (store if store is not None
                      else MovieStore(db_path or get_db_path()))
        self.data_dir = data_dir or get_data_dir()
        self.key_missing = False
        if client is not None:
            self.client = client
        else:
            try:
                self.client = TmdbClient(store=self.store)
            except RuntimeError:
                self.client = None
                self.key_missing = True
        self.discovery = (DiscoveryService(self.client, self.store)
                          if self.client is not None else None)
        self.builder = WatchlistBuilder(self.store, self.client)
        self.running = False

    # ---------- display ----------

    def display_welcome_message(self):
        """Show the welcome banner."""
        self.output_func(WELCOME_MESSAGE)

    def display_menu(self):
        """Show the list of available commands."""
        self.output_func(MENU_TEXT)

    # ---------- input ----------

    def get_command_input(self):
        """Read one command line and normalise it (strip + lowercase)."""
        raw = self.input_func("movie> ")
        return raw.strip().lower()

    # ---------- main loop ----------

    def run(self):
        """Main loop: read -> validate -> dispatch, without ever crashing."""
        self.display_welcome_message()
        self.display_menu()
        self.running = True
        while self.running:
            try:
                self.handle_command(self.get_command_input())
            except EOFError:
                self.output_func("")
                self.output_func("[สิ้นสุดอินพุต]")
                self.running = False
            except KeyboardInterrupt:
                self.output_func("")
                self.output_func("[ยกเลิกโดยผู้ใช้]")
                self.running = False
            except ValueError as exc:
                self.output_func(f"คำเตือน: {exc}")
            except TmdbError as exc:
                self.output_func(self.describe_tmdb_error(exc))
            except ExportError as exc:
                self.output_func(f"ไม่สามารถเขียนไฟล์ได้: {exc}")
        self.output_func(FAREWELL_MESSAGE)

    def handle_command(self, text):
        """Validate and dispatch one normalised command line."""
        if not text:
            raise ValueError("กรุณาพิมพ์คำสั่ง (พิมพ์ help เพื่อดูคำสั่งทั้งหมด)")
        parts = text.split()
        name, args = parts[0], parts[1:]
        if name in QUIT_COMMANDS:
            self.running = False
            return
        handlers = {
            "search": self.cmd_search,
            "discover": self.cmd_discover,
            "watchlist": self.cmd_watchlist,
            "favorites": self.cmd_favorites,
            "export": self.cmd_export,
            "history": self.cmd_history,
            "help": self.cmd_help,
        }
        handler = handlers.get(name)
        if handler is None:
            raise ValueError(
                f"ไม่รู้จักคำสั่ง '{name}' (พิมพ์ help เพื่อดูคำสั่งทั้งหมด)"
            )
        handler(args)

    # ---------- command handlers ----------

    def cmd_help(self, args):
        """help — show the command menu again."""
        self.display_menu()

    def cmd_search(self, args):
        """search <ชื่อเรื่อง> — search TMDB (cached) and log the query."""
        if not args:
            raise ValueError("คำสั่ง search ต้องระบุชื่อเรื่อง เช่น search alpha")
        if self.client is None:
            self.output_func(MISSING_KEY_MESSAGE)
            return
        query = " ".join(args)
        results = self.discovery.search(query)
        if not results:
            self.output_func(f"ไม่พบภาพยนตร์ที่ตรงกับ '{query}'")
            return
        self.output_func(f"พบ {len(results)} เรื่อง (แสดงสูงสุด 10 รายการแรก):")
        for movie in results:
            self.output_func("  " + self.format_movie(movie))

    def cmd_discover(self, args):
        """discover <รหัส> [ตัวเลือก] — similar movies, filtered and sorted."""
        if not args:
            raise ValueError(
                "คำสั่ง discover ต้องระบุรหัสภาพยนตร์ เช่น discover 101"
            )
        seed_id = self.parse_positive_int(args[0], "รหัสภาพยนตร์")
        options = self.parse_discover_options(args[1:])
        if self.client is None:
            self.output_func(MISSING_KEY_MESSAGE)
            return
        try:
            seed = self.builder.ensure_movie(seed_id)
        except TmdbError as exc:
            if "404" in str(exc):
                self.output_func(f"ไม่พบรหัสภาพยนตร์ {seed_id} ใน TMDB")
                return
            raise
        results = self.discovery.find_similar(
            seed_id,
            min_rating=options["min_rating"],
            min_votes=options["min_votes"],
            year_from=options["year_from"],
            year_to=options["year_to"],
            limit=options["limit"],
            sort=options["sort"],
        )
        if not results:
            self.output_func(
                f"ไม่พบภาพยนตร์ที่ผ่านเงื่อนไขสำหรับรหัส {seed_id}"
            )
            return
        self.output_func(
            f"ภาพยนตร์ที่คล้ายกับ '{seed['title']}' "
            f"({len(results)} เรื่อง · เรียงตาม {options['sort']}):"
        )
        for rank, movie in enumerate(results, start=1):
            self.output_func(f"  {rank}. {self.format_movie(movie)}")

    def cmd_watchlist(self, args):
        """watchlist add <รหัส> | list | remove <รหัส> | clear."""
        if not args:
            raise ValueError(
                "คำสั่ง watchlist ต้องมีคำสั่งย่อย: "
                "add <รหัส> | list | remove <รหัส> | clear"
            )
        action, rest = args[0], args[1:]
        if action == "list":
            self.show_list("watchlist")
        elif action == "add":
            self.add_to_list("watchlist", rest)
        elif action == "remove":
            self.remove_from_list("watchlist", rest)
        elif action == "clear":
            count = self.store.clear_watchlist()
            self.output_func(f"ล้างรายการรับชมเรียบร้อยแล้ว ({count} เรื่อง)")
        else:
            raise ValueError(
                f"ไม่รู้จักคำสั่งย่อย '{action}' "
                "(ใช้ add <รหัส> | list | remove <รหัส> | clear)"
            )

    def cmd_favorites(self, args):
        """favorites add <รหัส> | list | remove <รหัส>."""
        if not args:
            raise ValueError(
                "คำสั่ง favorites ต้องมีคำสั่งย่อย: "
                "add <รหัส> | list | remove <รหัส>"
            )
        action, rest = args[0], args[1:]
        if action == "list":
            self.show_list("favorites")
        elif action == "add":
            self.add_to_list("favorites", rest)
        elif action == "remove":
            self.remove_from_list("favorites", rest)
        else:
            raise ValueError(
                f"ไม่รู้จักคำสั่งย่อย '{action}' "
                "(ใช้ add <รหัส> | list | remove <รหัส>)"
            )

    def cmd_export(self, args):
        """export watchlist|favorites — write the list to CSV under data/."""
        if not args or args[0] not in LIST_LABELS:
            raise ValueError(
                "คำสั่ง export ต้องระบุรายการ: "
                "export watchlist หรือ export favorites"
            )
        kind = args[0]
        path = os.path.join(self.data_dir, f"{kind}.csv")
        if kind == "watchlist":
            count = Exporter.export_watchlist(self.store, path)
        else:
            count = Exporter.export_favorites(self.store, path)
        self.output_func(f"ส่งออก {count} รายการไปที่ {path}")

    def cmd_history(self, args):
        """history [N] — show the most recent searches (default 10)."""
        limit = 10
        if args:
            limit = self.parse_positive_int(args[0], "จำนวนรายการ")
        rows = self.store.get_recent_searches(limit)
        if not rows:
            self.output_func("ยังไม่มีประวัติการค้นหา")
            return
        self.output_func(f"ประวัติการค้นหาล่าสุด ({len(rows)} รายการ):")
        for index, row in enumerate(rows, start=1):
            self.output_func(
                f"  {index}. '{row['query']}' — พบ {row['result_count']} ผล "
                f"({row['searched_at']})"
            )

    # ---------- list helpers ----------

    def show_list(self, kind):
        """Print one stored list (``watchlist`` or ``favorites``)."""
        label = LIST_LABELS[kind]
        rows = (self.store.get_watchlist() if kind == "watchlist"
                else self.store.get_favorites())
        if not rows:
            self.output_func(f"{label}ยังว่างอยู่ (ใช้ {kind} add <รหัส>)")
            return
        self.output_func(f"{label} ({len(rows)} เรื่อง):")
        for row in rows:
            self.output_func("  " + self.format_movie(row))

    def add_to_list(self, kind, rest):
        """Add one movie id to the watchlist or favorites (fetch if needed)."""
        label = LIST_LABELS[kind]
        if not rest:
            raise ValueError(
                f"คำสั่ง {kind} add ต้องระบุรหัสภาพยนตร์ เช่น {kind} add 101"
            )
        movie_id = self.parse_positive_int(rest[0], "รหัสภาพยนตร์")
        if self.client is None and self.store.get_movie(movie_id) is None:
            self.output_func(MISSING_KEY_MESSAGE)
            return
        try:
            movie = self.builder.ensure_movie(movie_id)
        except TmdbError as exc:
            if "404" in str(exc):
                self.output_func(f"ไม่พบรหัสภาพยนตร์ {movie_id} ใน TMDB")
                return
            raise
        if kind == "watchlist":
            added = self.builder.add(movie_id)
        else:
            added = self.builder.add_favorite(movie_id)
        if added:
            self.output_func(f"เพิ่ม '{movie['title']}' เข้า{label}แล้ว")
        else:
            self.output_func(f"'{movie['title']}' อยู่ใน{label}แล้ว")

    def remove_from_list(self, kind, rest):
        """Remove one movie id from the watchlist or favorites."""
        label = LIST_LABELS[kind]
        if not rest:
            raise ValueError(
                f"คำสั่ง {kind} remove ต้องระบุรหัสภาพยนตร์ เช่น "
                f"{kind} remove 101"
            )
        movie_id = self.parse_positive_int(rest[0], "รหัสภาพยนตร์")
        if kind == "watchlist":
            removed = self.store.remove_from_watchlist(movie_id)
        else:
            removed = self.store.remove_favorite(movie_id)
        if removed:
            movie = self.store.get_movie(movie_id)
            title = movie["title"] if movie else str(movie_id)
            self.output_func(f"นำ '{title}' ออกจาก{label}แล้ว")
        else:
            self.output_func(f"ไม่พบรหัสภาพยนตร์ {movie_id} ใน{label}")

    # ---------- option / value parsing ----------

    def parse_discover_options(self, tokens):
        """Parse discover flags (--min-rating ... --limit) into a dict."""
        options = {"min_rating": 0.0, "min_votes": 0, "year_from": None,
                   "year_to": None, "sort": "rating", "limit": 10}
        index = 0
        while index < len(tokens):
            flag = tokens[index]
            if flag not in DISCOVER_FLAGS:
                raise ValueError(
                    f"ไม่รู้จักตัวเลือก '{flag}' "
                    f"(ใช้ {' · '.join(DISCOVER_FLAGS)})"
                )
            if index + 1 >= len(tokens):
                raise ValueError(f"ตัวเลือก {flag} ต้องมีค่าตามหลัง")
            value = tokens[index + 1]
            if flag == "--min-rating":
                options["min_rating"] = self._parse_rating(value)
            elif flag == "--min-votes":
                options["min_votes"] = self._parse_non_negative_int(value)
            elif flag == "--year":
                options["year_from"], options["year_to"] = \
                    self._parse_year_range(value)
            elif flag == "--sort":
                if value not in SORT_OPTIONS:
                    raise ValueError(
                        f"--sort ต้องเป็น {', '.join(SORT_OPTIONS)} "
                        f"(ได้รับ '{value}')"
                    )
                options["sort"] = value
            else:  # --limit
                options["limit"] = self.parse_positive_int(value, "จำนวนรายการ")
            index += 2
        return options

    @staticmethod
    def _parse_rating(text):
        """Convert ``text`` to a 0–10 rating for --min-rating."""
        try:
            value = float(text)
        except ValueError:
            raise ValueError(
                f"ค่า --min-rating ต้องเป็นตัวเลข (ได้รับ '{text}')"
            ) from None
        if not 0.0 <= value <= 10.0:
            raise ValueError("ค่า --min-rating ต้องอยู่ระหว่าง 0 ถึง 10")
        return value

    @staticmethod
    def _parse_non_negative_int(text):
        """Convert ``text`` to a whole number >= 0 for --min-votes."""
        try:
            value = int(text)
        except ValueError:
            raise ValueError(
                f"ค่า --min-votes ต้องเป็นจำนวนเต็ม (ได้รับ '{text}')"
            ) from None
        if value < 0:
            raise ValueError("ค่า --min-votes ต้องไม่ติดลบ")
        return value

    @staticmethod
    def _parse_year_range(text):
        """Convert 'YYYY' or 'YYYY-YYYY' to a (year_from, year_to) pair."""
        parts = text.split("-")
        if len(parts) == 1:
            parts = [parts[0], parts[0]]
        if len(parts) != 2 or not all(
            len(part) == 4 and part.isdigit() for part in parts
        ):
            raise ValueError(
                "ค่า --year ต้องเป็นปี 4 หลัก เช่น 2020 หรือช่วง 1990-2020 "
                f"(ได้รับ '{text}')"
            )
        year_from, year_to = int(parts[0]), int(parts[1])
        if year_from > year_to:
            raise ValueError("ช่วงปีต้องเรียงจากน้อยไปมาก เช่น 1990-2020")
        return year_from, year_to

    # ---------- shared helpers ----------

    @staticmethod
    def describe_tmdb_error(exc):
        """Map a TmdbError to a friendly Thai message."""
        message = str(exc)
        if "401" in message:
            return "คีย์ TMDB ไม่ถูกต้องหรือถูกปฏิเสธ (401) — ตรวจสอบค่า TMDB_API_KEY"
        if "429" in message:
            return "เรียกใช้ TMDB ถี่เกินไป (429) — รอสักครู่แล้วลองใหม่"
        if "404" in message:
            return "ไม่พบข้อมูลที่ขอจาก TMDB (404)"
        return f"เชื่อมต่อ TMDB ไม่สำเร็จ: {message}"

    @staticmethod
    def parse_positive_int(text, label):
        """Convert text to a positive integer or raise a readable error."""
        try:
            value = int(text)
        except ValueError:
            raise ValueError(
                f"{label}ต้องเป็นตัวเลข (ได้รับ '{text}')"
            ) from None
        if value <= 0:
            raise ValueError(f"{label}ต้องมากกว่า 0 (ได้รับ {value})")
        return value

    @staticmethod
    def format_movie(movie):
        """Format one movie dict as a single display line."""
        movie_id = movie.get("id")
        if movie_id is None:
            movie_id = movie.get("tmdb_id")
        year = (movie.get("release_date") or "")[:4] or "----"
        rating = movie.get("vote_average")
        rating_text = "-" if rating is None else f"{rating:.1f}"
        return f"[{movie_id}] {movie['title']} ({year}) คะแนน {rating_text}"


Writing src/cli.py


In [15]:
%%writefile src/app.py
"""Entry point for the interactive Movie Discovery CLI.

Run with:

    python -m src.app
"""

from .cli import CLI


def main():
    """Start the interactive CLI."""
    CLI().run()


if __name__ == "__main__":
    main()


Writing src/app.py


In [16]:
%%writefile web/__init__.py
"""Flask application factory for the web front-end (Sprint 3).

The web layer is a new presentation layer: it imports the Sprint 1-2
business and data services unchanged and turns page requests into calls
on the same objects the CLI uses.  ``client`` and ``store`` can be
injected so the test-suite and local checks run fully offline.
"""

import sqlite3

from flask import Flask, g, render_template

from src.cli import CLI
from src.config import get_data_dir, get_db_path
from src.discovery import DiscoveryService
from src.movie_store import MovieStore
from src.tmdb_client import TmdbClient, TmdbError
from src.watchlist import WatchlistBuilder


TMDB_ATTRIBUTION = (
    "This product uses the TMDB API but is not endorsed or certified by TMDB."
)

DB_HELP = (
    "วิธีแก้: ปิดเซิร์ฟเวอร์ · ย้ายหรือลบไฟล์ฐานข้อมูลด้านบน "
    "(ระบบจะสร้างไฟล์ใหม่ให้อัตโนมัติเมื่อเปิดเซิร์ฟเวอร์อีกครั้ง) "
    "แล้วเริ่มใช้งานใหม่"
)


def create_app(client=None, store=None, data_dir=None, db_path=None):
    """Build the web app; inject ``client``/``store`` to run offline (tests).

    Production mode builds one service bundle per request, each with its
    own SQLite connection: Python's sqlite3 refuses a connection created
    in one thread being used from another, and Flask's development server
    handles requests in worker threads.  Injected objects (tests) are
    reused as-is, keeping everything in the test thread and allowing an
    in-memory database to survive across requests.
    """
    app = Flask(__name__)
    app.config["DATA_DIR"] = data_dir or get_data_dir()
    app.config["DB_PATH"] = db_path or get_db_path()

    def database_error_page(exc):
        """Polite page for an unreadable database file (with repair steps)."""
        return render_template(
            "error.html",
            title="ฐานข้อมูลมีปัญหา",
            message=(
                f"เปิดไฟล์ฐานข้อมูลไม่สำเร็จ: {exc} · "
                f"ไฟล์: {app.config['DB_PATH']} — {DB_HELP}"
            ),
        ), 500

    def build_services():
        """Create the service bundle for one request."""
        if store is not None or client is not None:
            test_store = store if store is not None else MovieStore(":memory:")
            return {
                "store": test_store,
                "client": client,
                "discovery": (DiscoveryService(client, test_store)
                              if client is not None else None),
                "builder": WatchlistBuilder(test_store, client),
                "key_missing": client is None,
                "data_dir": app.config["DATA_DIR"],
                "close_store": False,
            }
        request_store = MovieStore(app.config["DB_PATH"])
        request_client = None
        key_missing = False
        try:
            request_client = TmdbClient(store=request_store)
        except RuntimeError:
            key_missing = True
        return {
            "store": request_store,
            "client": request_client,
            "discovery": (DiscoveryService(request_client, request_store)
                          if request_client is not None else None),
            "builder": WatchlistBuilder(request_store, request_client),
            "key_missing": key_missing,
            "data_dir": app.config["DATA_DIR"],
            "close_store": True,
        }

    @app.before_request
    def load_services():
        """Prepare the per-request services before every page.

        A damaged database file fails here (connection + table setup);
        the request stops early with a polite repair page instead of a
        raw traceback.
        """
        try:
            g.services = build_services()
        except sqlite3.Error as exc:
            g.services = None
            return database_error_page(exc)

    @app.teardown_request
    def close_services(exc):
        """Close the request's SQLite connection when the request ends."""
        bundle = g.pop("services", None)
        if bundle is not None and bundle["close_store"]:
            bundle["store"].close()

    from .routes import bp
    app.register_blueprint(bp)

    @app.errorhandler(TmdbError)
    def handle_tmdb_error(exc):
        """Map TMDB failures to a polite error page (CLI wording reused)."""
        return render_template("error.html",
                               title="เชื่อมต่อ TMDB ไม่สำเร็จ",
                               message=CLI.describe_tmdb_error(exc)), 502

    @app.errorhandler(sqlite3.Error)
    def handle_database_error(exc):
        """Fallback page when the database fails during a request."""
        return database_error_page(exc)

    @app.errorhandler(404)
    def handle_not_found(exc):
        """Show a friendly page for unknown URLs."""
        return render_template(
            "error.html",
            title="ไม่พบหน้าที่ต้องการ (404)",
            message="ลิงก์อาจไม่ถูกต้อง หรือหน้านี้ยังไม่เปิดใช้งาน",
        ), 404

    @app.context_processor
    def inject_nav_state():
        """Give every page the nav list counts and the key-banner state."""
        bundle = getattr(g, "services", None)
        if bundle is None:
            # Database error path: neutral navigation, no key banner.
            return {
                "nav_watchlist": 0,
                "nav_favorites": 0,
                "key_missing": False,
                "tmdb_attribution": TMDB_ATTRIBUTION,
            }
        return {
            "nav_watchlist": len(bundle["store"].get_watchlist()),
            "nav_favorites": len(bundle["store"].get_favorites()),
            "key_missing": bundle["key_missing"],
            "tmdb_attribution": TMDB_ATTRIBUTION,
        }

    return app


Writing web/__init__.py


In [17]:
%%writefile web/routes.py
"""Page routes for the web front-end (Sprint 3).

Implemented so far: the home page, the search results page, the movie
details page (similar movies, filters and list buttons), the two list
pages (cross-list stars, CSV download) and the search history.
"""

import os

from flask import (Blueprint, abort, g, redirect, render_template, request,
                   send_file)

from src.cli import CLI, LIST_LABELS
from src.discovery import SORT_OPTIONS
from src.exporter import Exporter, ExportError
from src.tmdb_client import TmdbError

bp = Blueprint("main", __name__)

SORT_LABELS = {
    "rating": "คะแนน",
    "votes": "จำนวนโหวต",
    "year": "ปีใหม่ก่อน",
    "popularity": "ความนิยม",
}

# Messages shown after an add/remove action (Post -> Redirect -> Get).
ACTION_NOTICES = {
    ("added", "watchlist"): "เพิ่มเข้าสู่รายการรับชมแล้ว",
    ("added", "favorites"): "เพิ่มเข้าสู่รายการโปรดแล้ว",
    ("exists", "watchlist"): "เรื่องนี้อยู่ในรายการรับชมอยู่แล้ว",
    ("exists", "favorites"): "เรื่องนี้อยู่ในรายการโปรดอยู่แล้ว",
    ("removed", "watchlist"): "ลบออกจากรายการรับชมแล้ว",
    ("removed", "favorites"): "ลบออกจากรายการโปรดแล้ว",
    ("missing", "watchlist"): "ไม่พบเรื่องนี้ในรายการรับชม (อาจถูกลบไปแล้ว)",
    ("missing", "favorites"): "ไม่พบเรื่องนี้ในรายการโปรด (อาจถูกลบไปแล้ว)",
    ("error", "nodata"): (
        "เพิ่มไม่สำเร็จ: ไม่มีข้อมูลหนังในเครื่อง "
        "และโหมดนี้ไม่มีคีย์ TMDB"
    ),
}


def get_services():
    """Return the per-request service bundle (ready before every request)."""
    return g.services


def _notice_from_args():
    """Pick up the action message a redirect may have attached."""
    for (key, kind), text in ACTION_NOTICES.items():
        if request.args.get(key) == kind:
            return text
    return None


def _with(path, param):
    """Append one query parameter to a local path (? or & as needed)."""
    separator = "&" if "?" in path else "?"
    return f"{path}{separator}{param}"


def _safe_target(default):
    """Return the form's ``next`` path when it is a local URL.

    Rejects absolute URLs and protocol-relative tricks (``//host`` and
    ``/\\host``), so the redirect can never leave this application.
    """
    target = (request.form.get("next") or "").strip()
    if ("\\" in target or not target.startswith("/")
            or target.startswith("//")):
        return default
    return target


def _parse_list_filters(args):
    """Read the similar-list filters; invalid values fall back + note."""
    filters = {"min_rating": 0.0, "min_votes": 0, "year_from": None,
               "year_to": None, "sort": "rating", "limit": 10}
    notes = []

    def read_number(name, cast, low, high, label):
        """Parse one numeric field, or fall back with a note."""
        raw = (args.get(name) or "").strip()
        if not raw:
            return None, None
        try:
            value = cast(raw)
        except ValueError:
            return None, f"ค่า{label}ไม่ใช่ตัวเลข — ใช้ค่าเริ่มต้นแทน"
        if not low <= value <= high:
            return None, f"ค่า{label}อยู่นอกช่วงที่รองรับ — ใช้ค่าเริ่มต้นแทน"
        return value, None

    value, note = read_number("min_rating", float, 0.0, 10.0, "คะแนนขั้นต่ำ")
    if note:
        notes.append(note)
    else:
        filters["min_rating"] = value if value is not None else 0.0

    value, note = read_number("min_votes", int, 0, 10 ** 7, "โหวตขั้นต่ำ")
    if note:
        notes.append(note)
    else:
        filters["min_votes"] = value if value is not None else 0

    value, note = read_number("year_from", int, 1900, 2100, "ปีเริ่มต้น")
    if note:
        notes.append(note)
    else:
        filters["year_from"] = value

    value, note = read_number("year_to", int, 1900, 2100, "ปีสิ้นสุด")
    if note:
        notes.append(note)
    else:
        filters["year_to"] = value

    if (filters["year_from"] and filters["year_to"]
            and filters["year_from"] > filters["year_to"]):
        filters["year_from"], filters["year_to"] = (
            filters["year_to"], filters["year_from"])
        notes.append("ช่วงปีสลับกัน — สลับให้จากน้อยไปมากแล้ว")

    sort = (args.get("sort") or "").strip()
    if sort:
        if sort in SORT_OPTIONS:
            filters["sort"] = sort
        else:
            notes.append("ไม่รู้จักวิธีเรียงที่เลือก — ใช้ 'rating' แทน")

    value, note = read_number("limit", int, 1, 20, "จำนวนรายการ")
    if note:
        notes.append(note)
    else:
        filters["limit"] = value if value is not None else 10

    return filters, notes


def _movie_not_found(tmdb_id):
    """Render the polite 'movie not found' page for a bad TMDB id."""
    return render_template(
        "error.html",
        title="ไม่พบภาพยนตร์เรื่องนี้ (404)",
        message=f"รหัสภาพยนตร์ {tmdb_id} ไม่มีใน TMDB — "
                "ตรวจสอบรหัสแล้วลองใหม่",
    ), 404


def _add_to_list(kind, tmdb_id):
    """Shared handler: add by id, then redirect back with a result flag."""
    builder = get_services()["builder"]
    try:
        if kind == "watchlist":
            added = builder.add(tmdb_id)
        else:
            added = builder.add_favorite(tmdb_id)
    except TmdbError as exc:
        if "404" in str(exc):
            return _movie_not_found(tmdb_id)
        raise
    except ValueError:
        target = _safe_target(f"/movie/{tmdb_id}")
        return redirect(_with(target, "error=nodata"), code=303)
    flag = "added" if added else "exists"
    target = _safe_target(f"/movie/{tmdb_id}")
    return redirect(_with(target, f"{flag}={kind}"), code=303)


def _remove_from_list(kind, tmdb_id):
    """Shared handler: remove by id, then redirect back with a flag."""
    store = get_services()["store"]
    if kind == "watchlist":
        removed = store.remove_from_watchlist(tmdb_id)
    else:
        removed = store.remove_favorite(tmdb_id)
    flag = "removed" if removed else "missing"
    target = _safe_target(f"/movie/{tmdb_id}")
    return redirect(_with(target, f"{flag}={kind}"), code=303)


def _render_list(kind):
    """Render one list page (watchlist or favorites)."""
    services = get_services()
    store = services["store"]
    if kind == "watchlist":
        rows = store.get_watchlist()
        other_ids = {row["tmdb_id"] for row in store.get_favorites()}
        other_label = LIST_LABELS["favorites"]
    else:
        rows = store.get_favorites()
        other_ids = {row["tmdb_id"] for row in store.get_watchlist()}
        other_label = LIST_LABELS["watchlist"]
    notices = []
    notice = _notice_from_args()
    if notice:
        notices.append(notice)
    if request.args.get("cleared") == "watchlist":
        count = request.args.get("count", "0")
        notices.append(f"ล้างรายการรับชมเรียบร้อยแล้ว ({count} เรื่อง)")
    return render_template(
        "list.html",
        kind=kind,
        page_title=LIST_LABELS[kind],
        rows=rows,
        other_ids=other_ids,
        other_label=other_label,
        notices=notices,
    )


@bp.get("/")
def home():
    """Home page: search form and a short introduction."""
    return render_template("home.html")


@bp.get("/search")
def search():
    """Run the query through the discovery service and show movie cards."""
    services = get_services()
    query = (request.args.get("q") or "").strip()
    if not query:
        return render_template(
            "search.html",
            query="",
            notice="กรุณาพิมพ์คำค้นก่อนกดค้นหา",
        )
    notice = None
    max_length = 120  # กันคำค้นยาวผิดปกติ — ค้นเฉพาะส่วนแรก
    if len(query) > max_length:
        query = query[:max_length]
        notice = f"คำค้นยาวเกิน {max_length} ตัวอักษร — ค้นเฉพาะส่วนแรกให้"
    if services["client"] is None:
        return render_template(
            "search.html",
            query=query,
            notice=("ยังไม่ได้ตั้งค่าคีย์ TMDB — กำหนดตัวแปรสภาพแวดล้อม "
                    "TMDB_API_KEY แล้วเปิดเซิร์ฟเวอร์ใหม่ "
                    "(หน้าออฟไลน์ เช่นรายการรับชม · ประวัติ ยังใช้ได้ตามปกติ)"),
        )
    results = services["discovery"].search(query)
    return render_template("search.html", query=query,
                           results=results, notice=notice)


@bp.get("/movie/<int:tmdb_id>")
def movie(tmdb_id):
    """Movie details, similar movies (filterable) and list buttons."""
    services = get_services()
    client = services["client"]
    store = services["store"]
    notices = []
    notice = _notice_from_args()
    if notice:
        notices.append(notice)

    if client is not None:
        try:
            movie = client.get_movie_details(tmdb_id)
        except TmdbError as exc:
            if "404" in str(exc):
                return _movie_not_found(tmdb_id)
            raise
        store.add_movie(movie)
    else:
        movie = store.get_movie(tmdb_id)
        if movie is None:
            return render_template(
                "error.html",
                title="ไม่มีข้อมูลภาพยนตร์เรื่องนี้ในเครื่อง",
                message=("โหมดไม่มีคีย์ TMDB จะดูได้เฉพาะเรื่องที่บันทึกไว้แล้ว "
                         "— เปิดโหมดมีคีย์เพื่อค้นหาเรื่องใหม่"),
            ), 404
        notices.append("โหมดไม่มีคีย์ TMDB: แสดงข้อมูลที่บันทึกไว้ในเครื่อง")

    movie = dict(movie)
    movie.setdefault("id", tmdb_id)
    poster_path = movie.get("poster_path")
    backdrop_path = movie.get("backdrop_path")
    poster_url = (f"https://image.tmdb.org/t/p/w342{poster_path}"
                  if poster_path else None)
    backdrop_url = (f"https://image.tmdb.org/t/p/w780{backdrop_path}"
                    if backdrop_path else None)

    filters, notes = _parse_list_filters(request.args)
    notices.extend(notes)

    similar = []
    similar_error = None
    if client is None:
        for link in store.get_similar_links(tmdb_id):
            stored = store.get_movie(link["target_id"])
            if stored:
                stored.setdefault("id", link["target_id"])
                similar.append(stored)
    else:
        try:
            similar = services["discovery"].find_similar(
                tmdb_id,
                min_rating=filters["min_rating"],
                min_votes=filters["min_votes"],
                year_from=filters["year_from"],
                year_to=filters["year_to"],
                limit=filters["limit"],
                sort=filters["sort"],
            )
        except TmdbError as exc:
            similar_error = CLI.describe_tmdb_error(exc)

    year = (movie.get("release_date") or "")[:4] or None
    return render_template(
        "movie.html",
        movie=movie,
        year=year,
        poster_url=poster_url,
        backdrop_url=backdrop_url,
        genres=movie.get("genres") or [],
        runtime=movie.get("runtime"),
        filters=filters,
        sorts=SORT_OPTIONS,
        sort_labels=SORT_LABELS,
        notices=notices,
        similar=similar,
        similar_error=similar_error,
        in_watchlist=store.is_in_watchlist(tmdb_id),
        in_favorites=store.is_in_favorites(tmdb_id),
    )


@bp.get("/watchlist")
def watchlist():
    """Show the watchlist with cross-list stars and CSV download."""
    return _render_list("watchlist")


@bp.get("/favorites")
def favorites():
    """Show favorites with cross-list stars and CSV download."""
    return _render_list("favorites")


@bp.get("/history")
def history():
    """Show the most recent searches (newest first)."""
    limit_raw = (request.args.get("limit") or "").strip()
    limit = 10
    if limit_raw:
        try:
            limit = max(1, min(int(limit_raw), 50))
        except ValueError:
            limit = 10
    rows = get_services()["store"].get_recent_searches(limit)
    return render_template("history.html", rows=rows, limit=limit)


@bp.get("/export/<kind>")
def export_list(kind):
    """Download the current list as a CSV file (written by Exporter)."""
    if kind not in LIST_LABELS:
        abort(404)
    services = get_services()
    path = os.path.join(services["data_dir"], f"{kind}.csv")
    try:
        if kind == "watchlist":
            Exporter.export_watchlist(services["store"], path)
        else:
            Exporter.export_favorites(services["store"], path)
    except ExportError as exc:
        return render_template(
            "error.html",
            title="สร้างไฟล์ CSV ไม่สำเร็จ",
            message=f"เขียนไฟล์ไม่สำเร็จ: {exc}",
        ), 500
    return send_file(path, as_attachment=True,
                     download_name=f"{kind}.csv", mimetype="text/csv")


@bp.post("/watchlist/add/<int:tmdb_id>")
def watchlist_add(tmdb_id):
    """Add one movie to the watchlist, then return to the page."""
    return _add_to_list("watchlist", tmdb_id)


@bp.post("/watchlist/remove/<int:tmdb_id>")
def watchlist_remove(tmdb_id):
    """Remove one movie from the watchlist, then return to the page."""
    return _remove_from_list("watchlist", tmdb_id)


@bp.post("/watchlist/clear")
def watchlist_clear():
    """Clear the whole watchlist, then return with a count notice."""
    count = get_services()["store"].clear_watchlist()
    target = _safe_target("/watchlist")
    return redirect(_with(target, f"cleared=watchlist&count={count}"),
                    code=303)


@bp.post("/favorites/add/<int:tmdb_id>")
def favorites_add(tmdb_id):
    """Add one movie to favorites, then return to the page."""
    return _add_to_list("favorites", tmdb_id)


@bp.post("/favorites/remove/<int:tmdb_id>")
def favorites_remove(tmdb_id):
    """Remove one movie from favorites, then return to the page."""
    return _remove_from_list("favorites", tmdb_id)


Writing web/routes.py


In [18]:
%%writefile web/__main__.py
"""Run the local development server: ``python -m web``.

Optional environment variables:
  PORT       listen port (default 5000)
  WEB_DEBUG  set to 1 to enable Flask debug/reload mode
"""

import os

from web import create_app

app = create_app()

if __name__ == "__main__":
    app.run(host="127.0.0.1",
            port=int(os.environ.get("PORT", "5000")),
            debug=os.environ.get("WEB_DEBUG") == "1")


Writing web/__main__.py


In [19]:
%%writefile web/templates/base.html
<!doctype html>
<html lang="th">
<head>
<meta charset="utf-8">
<meta name="viewport" content="width=device-width, initial-scale=1">
<title>{% block title %}Movie Discovery{% endblock %}</title>
<link rel="stylesheet" href="{{ url_for('static', filename='style.css') }}">
</head>
<body>
<header class="topbar">
  <a class="brand" href="{{ url_for('main.home') }}">Movie Discovery</a>
  <nav>
    <a class="nav-stat" href="/">หน้าแรก</a>
    <a class="nav-stat" href="/watchlist">รายการรับชม ({{ nav_watchlist }})</a>
    <a class="nav-stat" href="/favorites">รายการโปรด ({{ nav_favorites }})</a>
    <a class="nav-stat" href="/history">ประวัติ</a>
  </nav>
</header>

{% if key_missing %}
<div class="banner">
  ยังไม่ได้ตั้งค่าคีย์ TMDB — กำหนดตัวแปรสภาพแวดล้อม <code>TMDB_API_KEY</code>
  แล้วเปิดเซิร์ฟเวอร์ใหม่ · สร้างคีย์ฟรีได้ที่ developer.themoviedb.org —
  ส่วนรายการรับชม · รายการโปรด · ประวัติ ใช้งานได้ตามปกติ
</div>
{% endif %}

<main>
{% block content %}{% endblock %}
</main>

<footer class="footer">{{ tmdb_attribution }}</footer>
</body>
</html>


Writing web/templates/base.html


In [20]:
%%writefile web/templates/_card.html
{# One movie card: poster, title, year and rating — shared by every page. #}
{% macro movie_card(movie) -%}
<a class="card" href="/movie/{{ movie.id }}">
  <div class="poster">
    {% if movie.poster_path %}
    <img src="https://image.tmdb.org/t/p/w200{{ movie.poster_path }}"
         alt="โปสเตอร์ {{ movie.title }}" loading="lazy">
    {% else %}
    <span class="poster-fallback">🎬</span>
    {% endif %}
  </div>
  <div class="card-body">
    <div class="card-title">{{ movie.title }}</div>
    <div class="card-meta">
      {{ movie.release_date[:4] if movie.release_date else '----' }}
      · คะแนน {{ '%.1f'|format(movie.vote_average)
                 if movie.vote_average is not none else '-' }}
    </div>
  </div>
</a>
{%- endmacro %}


Writing web/templates/_card.html


In [21]:
%%writefile web/templates/home.html
{% extends "base.html" %}
{% block title %}Movie Discovery &amp; Watchlist Builder{% endblock %}
{% block content %}
<section class="hero">
  <h1>ระบบค้นพบภาพยนตร์และสร้างรายการรับชม</h1>
  <p>ค้นหาภาพยนตร์จาก TMDB สำรวจเรื่องที่คล้ายกัน จัดการรายการรับชมของคุณ
     และรับชมภาพยนตร์สาธารณสมบัติได้จากในตัวเว็บ</p>
  <form class="search-form" action="/search" method="get">
    <input type="text" name="q" placeholder="ค้นหาชื่อภาพยนตร์ เช่น inception"
           autocomplete="off">
    <button type="submit">ค้นหา</button>
  </form>
  <p class="quick-links">
    ไปที่: <a href="/watchlist">รายการรับชม</a> ·
    <a href="/favorites">รายการโปรด</a> ·
    <a href="/history">ประวัติการค้นหา</a>
  </p>
</section>
{% endblock %}


Writing web/templates/home.html


In [22]:
%%writefile web/templates/search.html
{% extends "base.html" %}
{% block title %}ค้นหา — Movie Discovery{% endblock %}
{% block content %}
<section class="search-panel">
  <form class="search-form" action="/search" method="get">
    <input type="text" name="q" value="{{ query }}"
           placeholder="ค้นหาชื่อภาพยนตร์ เช่น inception" autocomplete="off">
    <button type="submit">ค้นหา</button>
  </form>
</section>

{% if notice %}
<p class="notice">{{ notice }}</p>
{% endif %}

{% if results is defined %}
  {% if results %}
  <h2 class="result-count">พบ {{ results|length }} เรื่อง (แสดงสูงสุด 10 รายการแรก):</h2>
  {% import "_card.html" as cards with context %}
  <div class="cards">
    {% for movie in results %}{{ cards.movie_card(movie) }}{% endfor %}
  </div>
  {% else %}
  <p class="notice">ไม่พบภาพยนตร์ที่ตรงกับ '{{ query }}'</p>
  {% endif %}
{% endif %}
{% endblock %}


Writing web/templates/search.html


In [23]:
%%writefile web/templates/movie.html
{% extends "base.html" %}
{% import "_card.html" as cards with context %}
{% block title %}{{ movie.title }} — Movie Discovery{% endblock %}
{% block content %}

{% for text in notices %}
<p class="notice">{{ text }}</p>
{% endfor %}

<section class="detail-hero"
    {% if backdrop_url %}
    style="background-image: linear-gradient(rgba(13, 27, 52, 0.72),
           rgba(13, 27, 52, 0.88)), url('{{ backdrop_url }}');"
    {% endif %}>
  <div class="detail-poster">
    {% if poster_url %}
    <img src="{{ poster_url }}" alt="โปสเตอร์ {{ movie.title }}">
    {% else %}
    <span class="poster-fallback">🎬</span>
    {% endif %}
  </div>
  <div class="detail-info">
    <h1>{{ movie.title }}</h1>
    <p class="detail-meta">
      {{ year or '----' }}
      {% if runtime %} · {{ runtime }} นาที{% endif %}
      {% if movie.vote_average is not none %}
      · คะแนน ★ {{ '%.1f'|format(movie.vote_average) }} / 10
      {% endif %}
      {% if movie.vote_count %}
      · {{ '{:,}'.format(movie.vote_count) }} โหวต
      {% endif %}
    </p>
    {% if genres %}
    <p class="genres">
      {% for genre in genres %}<span class="chip">{{ genre.name }}</span>{% endfor %}
    </p>
    {% endif %}
    {% if movie.overview %}
    <p class="overview">{{ movie.overview }}</p>
    {% else %}
    <p class="overview muted">(ไม่มีเรื่องย่อในฐานข้อมูล)</p>
    {% endif %}
    <div class="actions">
      {% if in_watchlist %}
      <form method="post" action="/watchlist/remove/{{ movie.id }}">
        <input type="hidden" name="next" value="/movie/{{ movie.id }}">
        <button class="btn ghost" type="submit">− ลบออกจากรายการรับชม</button>
      </form>
      {% else %}
      <form method="post" action="/watchlist/add/{{ movie.id }}">
        <input type="hidden" name="next" value="/movie/{{ movie.id }}">
        <button class="btn" type="submit">+ เพิ่มเข้ารายการรับชม</button>
      </form>
      {% endif %}
      {% if in_favorites %}
      <form method="post" action="/favorites/remove/{{ movie.id }}">
        <input type="hidden" name="next" value="/movie/{{ movie.id }}">
        <button class="btn ghost" type="submit">− ลบออกจากรายการโปรด</button>
      </form>
      {% else %}
      <form method="post" action="/favorites/add/{{ movie.id }}">
        <input type="hidden" name="next" value="/movie/{{ movie.id }}">
        <button class="btn" type="submit">+ เพิ่มเข้ารายการโปรด</button>
      </form>
      {% endif %}
    </div>
    {% if in_watchlist or in_favorites %}
    <p class="state-line">
      {% if in_watchlist %}<span class="state-on">✓ อยู่ในรายการรับชม</span>{% endif %}
      {% if in_favorites %}<span class="state-on">✓ อยู่ในรายการโปรด</span>{% endif %}
    </p>
    {% endif %}
  </div>
</section>

<section class="similar">
  <h2>หนังคล้ายกัน</h2>
  <form class="filters" method="get" action="/movie/{{ movie.id }}">
    <label>คะแนนขั้นต่ำ
      <input type="number" name="min_rating" min="0" max="10" step="0.5"
             value="{{ filters.min_rating }}">
    </label>
    <label>โหวตขั้นต่ำ
      <input type="number" name="min_votes" min="0" step="10"
             value="{{ filters.min_votes }}">
    </label>
    <label>ปีจาก
      <input type="number" name="year_from" min="1900" max="2100"
             value="{{ filters.year_from if filters.year_from else '' }}">
    </label>
    <label>ถึงปี
      <input type="number" name="year_to" min="1900" max="2100"
             value="{{ filters.year_to if filters.year_to else '' }}">
    </label>
    <label>เรียงตาม
      <select name="sort">
        {% for option in sorts %}
        <option value="{{ option }}"{% if option == filters.sort %} selected{% endif %}>{{ sort_labels[option] }}</option>
        {% endfor %}
      </select>
    </label>
    <label>จำนวน
      <select name="limit">
        {% for option in (5, 10, 15, 20) %}
        <option value="{{ option }}"{% if option == filters.limit %} selected{% endif %}>{{ option }}</option>
        {% endfor %}
      </select>
    </label>
    <button class="btn" type="submit">กรอง</button>
  </form>

  {% if similar_error %}
  <p class="notice">{{ similar_error }}</p>
  {% elif similar %}
  <div class="cards">
    {% for item in similar %}{{ cards.movie_card(item) }}{% endfor %}
  </div>
  {% else %}
  <p class="notice">ไม่พบหนังคล้ายที่ผ่านตัวกรอง — ลองลดเงื่อนไขแล้วกรองใหม่</p>
  {% endif %}
</section>
{% endblock %}


Writing web/templates/movie.html


In [24]:
%%writefile web/templates/list.html
{% extends "base.html" %}
{% block title %}{{ page_title }} — Movie Discovery{% endblock %}
{% block content %}
<h1 class="page-title">{{ page_title }} ({{ rows|length }})</h1>

{% for text in notices %}
<p class="notice">{{ text }}</p>
{% endfor %}

<div class="list-tools">
  <a class="btn" href="/export/{{ kind }}">ดาวน์โหลด CSV</a>
  {% if kind == "watchlist" and rows %}
  <form method="post" action="/watchlist/clear"
        onsubmit="return confirm('ล้างรายการรับชมทั้งหมด?');">
    <input type="hidden" name="next" value="/watchlist">
    <button class="btn danger" type="submit">ล้างทั้งหมด</button>
  </form>
  {% endif %}
</div>

{% if rows %}
<table class="list-table">
  <thead>
    <tr>
      <th>เรื่อง</th>
      <th>ปี</th>
      <th>คะแนน</th>
      <th>เพิ่มเมื่อ</th>
      <th></th>
    </tr>
  </thead>
  <tbody>
    {% for row in rows %}
    <tr>
      <td>
        <a href="/movie/{{ row.tmdb_id }}">{{ row.title or '(ไม่มีชื่อ)' }}</a>
        {% if row.tmdb_id in other_ids %}
        <span class="star" title="อยู่ใน{{ other_label }}ด้วย">★</span>
        {% endif %}
      </td>
      <td>{{ row.release_date[:4] if row.release_date else '----' }}</td>
      <td>{{ '%.1f'|format(row.vote_average)
                 if row.vote_average is not none else '-' }}</td>
      <td class="when">{{ row.added_at[:16] }}</td>
      <td>
        <form method="post" action="/{{ kind }}/remove/{{ row.tmdb_id }}">
          <input type="hidden" name="next" value="/{{ kind }}">
          <button class="btn tiny" type="submit">ลบ</button>
        </form>
      </td>
    </tr>
    {% endfor %}
  </tbody>
</table>
{% else %}
<p class="notice">ยังไม่มีรายการ — เพิ่มได้จากหน้าค้นหาหรือหน้ารายละเอียดหนัง</p>
{% endif %}
{% endblock %}


Writing web/templates/list.html


In [25]:
%%writefile web/templates/history.html
{% extends "base.html" %}
{% block title %}ประวัติการค้นหา — Movie Discovery{% endblock %}
{% block content %}
<h1 class="page-title">ประวัติการค้นหาล่าสุด</h1>

<form class="history-limit" method="get" action="/history">
  <label>แสดง
    <select name="limit">
      {% for option in (10, 25, 50) %}
      <option value="{{ option }}"{% if option == limit %} selected{% endif %}>{{ option }}</option>
      {% endfor %}
    </select>
  </label>
  <button class="btn tiny" type="submit">แสดง</button> รายการ
</form>

{% if rows %}
<table class="list-table">
  <thead>
    <tr>
      <th>#</th>
      <th>คำค้น</th>
      <th>ผลลัพธ์</th>
      <th>เมื่อ</th>
    </tr>
  </thead>
  <tbody>
    {% for row in rows %}
    <tr>
      <td>{{ loop.index }}</td>
      <td>{{ row.query }}</td>
      <td>{{ row.result_count }}</td>
      <td class="when">{{ row.searched_at[:16] }}</td>
    </tr>
    {% endfor %}
  </tbody>
</table>
{% else %}
<p class="notice">ยังไม่มีประวัติการค้นหา</p>
{% endif %}
{% endblock %}


Writing web/templates/history.html


In [26]:
%%writefile web/templates/error.html
{% extends "base.html" %}
{% block title %}{{ title }} — Movie Discovery{% endblock %}
{% block content %}
<section class="notice">
  <h1 class="error-title">{{ title }}</h1>
  <p>{{ message }}</p>
  <p><a class="btn" href="{{ url_for('main.home') }}">กลับหน้าแรก</a></p>
</section>
{% endblock %}


Writing web/templates/error.html


In [27]:
%%writefile web/static/style.css
:root {
  --ink: #1b1b1b;
  --navy: #1F3864;
  --line: #d9dce3;
  --bg: #f7f8fa;
}

* { box-sizing: border-box; }

body {
  margin: 0;
  font-family: "Segoe UI", Tahoma, sans-serif;
  color: var(--ink);
  background: var(--bg);
}

.topbar {
  display: flex;
  align-items: center;
  gap: 24px;
  padding: 14px 28px;
  background: #ffffff;
  border-bottom: 3px solid var(--navy);
}

.brand {
  font-weight: 700;
  color: var(--navy);
  text-decoration: none;
  font-size: 18px;
}

.topbar nav {
  display: flex;
  gap: 18px;
  margin-left: auto;
}

.nav-stat { color: #444444; font-size: 14px; }

.banner {
  background: #fff4d6;
  border-bottom: 1px solid #e6d29b;
  padding: 10px 28px;
  font-size: 14px;
}

main {
  max-width: 980px;
  margin: 0 auto;
  padding: 28px;
}

.hero h1 { color: var(--navy); margin: 0 0 8px; }

.search-form {
  display: flex;
  gap: 10px;
  margin-top: 18px;
  max-width: 560px;
}

.search-form input {
  flex: 1;
  padding: 12px 14px;
  font-size: 16px;
  border: 1px solid var(--line);
  border-radius: 8px;
}

.search-form button {
  padding: 12px 22px;
  font-size: 16px;
  border: 0;
  border-radius: 8px;
  background: var(--navy);
  color: #ffffff;
  cursor: pointer;
}

.footer {
  max-width: 980px;
  margin: 40px auto 24px;
  padding: 0 28px;
  color: #777777;
  font-size: 12px;
}

.search-panel { margin-bottom: 6px; }

.result-count { color: var(--navy); font-size: 18px; margin: 22px 0 2px; }

.cards {
  display: grid;
  grid-template-columns: repeat(auto-fill, minmax(158px, 1fr));
  gap: 16px;
  margin-top: 14px;
}

.card {
  background: #ffffff;
  border: 1px solid var(--line);
  border-radius: 10px;
  overflow: hidden;
  text-decoration: none;
  color: inherit;
}

.card:hover { box-shadow: 0 4px 14px rgba(31, 56, 100, 0.18); }

.poster {
  aspect-ratio: 2 / 3;
  background: #e8eaf0;
  display: flex;
  align-items: center;
  justify-content: center;
  font-size: 34px;
}

.poster img { width: 100%; height: 100%; object-fit: cover; display: block; }

.card-body { padding: 10px 12px 14px; }

.card-title { font-weight: 600; margin-bottom: 4px; line-height: 1.3; }

.card-meta { color: #555566; font-size: 13px; }

.notice {
  background: #ffffff;
  border: 1px solid var(--line);
  border-radius: 8px;
  padding: 12px 16px;
  margin-top: 14px;
}

.error-title { color: var(--navy); margin-top: 0; }

.btn {
  display: inline-block;
  padding: 10px 18px;
  background: var(--navy);
  color: #ffffff;
  text-decoration: none;
  border-radius: 8px;
}

/* --- movie detail page --- */

.detail-hero {
  display: flex;
  gap: 22px;
  padding: 22px;
  border-radius: 12px;
  background-color: var(--navy);
  color: #ffffff;
  background-size: cover;
  background-position: center;
}

.detail-poster {
  flex: 0 0 180px;
  aspect-ratio: 2 / 3;
  background: rgba(255, 255, 255, 0.12);
  border-radius: 10px;
  overflow: hidden;
  display: flex;
  align-items: center;
  justify-content: center;
  font-size: 40px;
}

.detail-poster img {
  width: 100%;
  height: 100%;
  object-fit: cover;
  display: block;
}

.detail-info h1 { margin: 0 0 6px; font-size: 26px; }

.detail-meta { color: #d8def0; margin: 0 0 10px; }

.genres { margin: 0 0 12px; }

.chip {
  display: inline-block;
  background: rgba(255, 255, 255, 0.16);
  border-radius: 999px;
  padding: 3px 12px;
  margin-right: 6px;
  font-size: 13px;
}

.overview { line-height: 1.6; margin: 0 0 16px; max-width: 62ch; }

.overview.muted { color: #b9c2dd; font-style: italic; }

.actions { display: flex; flex-wrap: wrap; gap: 10px; margin-bottom: 8px; }

.actions form { display: inline; }

.btn.ghost {
  background: transparent;
  border: 1px solid rgba(255, 255, 255, 0.7);
  color: #ffffff;
}

.state-line { margin: 4px 0 0; }

.state-on { color: #9fe6a8; margin-right: 14px; }

.similar { margin-top: 26px; }

.similar h2 { color: var(--navy); font-size: 20px; margin: 0 0 10px; }

.filters {
  display: flex;
  flex-wrap: wrap;
  align-items: flex-end;
  gap: 10px 14px;
  background: #ffffff;
  border: 1px solid var(--line);
  border-radius: 10px;
  padding: 12px 16px;
  margin-bottom: 14px;
}

.filters label {
  display: flex;
  flex-direction: column;
  gap: 4px;
  font-size: 13px;
  color: #445;
}

.filters input,
.filters select {
  padding: 7px 9px;
  border: 1px solid #c8cede;
  border-radius: 7px;
  font-size: 14px;
  min-width: 76px;
}

.filters button { margin-left: auto; }

/* --- list / history pages --- */

a.nav-stat { color: inherit; text-decoration: none; }

a.nav-stat:hover { text-decoration: underline; }

.page-title { color: var(--navy); font-size: 22px; margin: 4px 0 14px; }

.list-tools {
  display: flex;
  gap: 10px;
  margin-bottom: 14px;
}

.list-tools form { display: inline; }

.btn.tiny {
  padding: 5px 12px;
  font-size: 13px;
}

.btn.danger { background: #b00020; }

.list-table {
  width: 100%;
  border-collapse: collapse;
  background: #ffffff;
  border: 1px solid var(--line);
  border-radius: 10px;
  overflow: hidden;
}

.list-table th,
.list-table td {
  text-align: left;
  padding: 9px 12px;
  border-bottom: 1px solid var(--line);
  vertical-align: middle;
}

.list-table th {
  background: #eef1f7;
  color: var(--navy);
  font-size: 13px;
}

.list-table tr:last-child td { border-bottom: none; }

.list-table .when { color: #667; font-size: 13px; }

.star { color: #e8a300; margin-left: 4px; }

.history-limit { margin-bottom: 12px; }

.quick-links { margin-top: 10px; }

.poster { position: relative; }


Writing web/static/style.css


In [28]:
%%writefile tests/conftest.py
"""Pytest configuration: make ``src`` and shared test helpers importable."""

import pathlib
import sys

ROOT = pathlib.Path(__file__).resolve().parents[1]
for path in (ROOT, ROOT / "tests"):
    if str(path) not in sys.path:
        sys.path.insert(0, str(path))


Writing tests/conftest.py


In [29]:
%%writefile tests/fakes.py
"""Test doubles shared by the test-suite."""

from src.sample_data import SAMPLE_MOVIES
from src.tmdb_client import TmdbError


class FakeTmdbClient:
    """Offline stand-in for TmdbClient with canned responses.

    By default the similar list and the recommendation list overlap on
    movie 103, which exercises the dedupe logic.
    """

    def __init__(self, similar=None, recommendations=None):
        self._similar = list(
            similar if similar is not None else SAMPLE_MOVIES[:3]
        )
        self._recommendations = list(
            recommendations if recommendations is not None else SAMPLE_MOVIES[2:]
        )
        # Call counters so tests can prove when the network was avoided.
        self.search_calls = []
        self.detail_calls = []

    def get_similar(self, tmdb_id, page=1):
        return list(self._similar)

    def get_recommendations(self, tmdb_id, page=1):
        return list(self._recommendations)

    def search_movies(self, query, page=1):
        """Titles containing ``query`` (case-insensitive), like TMDB would."""
        self.search_calls.append(query)
        needle = query.lower()
        return [dict(movie) for movie in SAMPLE_MOVIES
                if needle in movie["title"].lower()]

    def get_movie_details(self, tmdb_id):
        """Details for a sample movie; a 404-style error for unknown ids."""
        self.detail_calls.append(tmdb_id)
        for movie in SAMPLE_MOVIES:
            if movie["id"] == tmdb_id:
                return dict(movie)
        raise TmdbError(
            f"TMDB resource not found (HTTP 404): /movie/{tmdb_id}"
        )


class FakeResponse:
    """Minimal requests.Response stand-in for the client tests."""

    def __init__(self, status_code=200, payload=None):
        self.status_code = status_code
        self._payload = payload

    def json(self):
        if isinstance(self._payload, Exception):
            raise self._payload
        return self._payload


Writing tests/fakes.py


In [30]:
%%writefile tests/test_tmdb_client.py
"""Unit tests for the TMDB client (network calls are mocked)."""

import json
from unittest.mock import Mock

import pytest
import requests

from fakes import FakeResponse
from src.movie_store import MovieStore
from src.tmdb_client import TmdbClient, TmdbError

SEARCH_PAYLOAD = {"results": [{"id": 1, "title": "Inception"},
                              {"id": 2, "title": "Interstellar"}]}


def make_client(session):
    return TmdbClient(api_key="test-key", base_url="https://api.test/3",
                      session=session)


def make_cached_client(session, store, api_key="test-key"):
    return TmdbClient(api_key=api_key, base_url="https://api.test/3",
                      session=session, store=store)


def test_search_movies_returns_results():
    session = Mock()
    session.get.return_value = FakeResponse(200, SEARCH_PAYLOAD)
    client = make_client(session)
    results = client.search_movies("inception")
    assert [movie["id"] for movie in results] == [1, 2]


def test_get_sends_api_key_language_and_query():
    session = Mock()
    session.get.return_value = FakeResponse(200, SEARCH_PAYLOAD)
    client = make_client(session)
    client.search_movies("inception")
    args, kwargs = session.get.call_args
    assert args[0] == "https://api.test/3/search/movie"
    assert kwargs["params"]["api_key"] == "test-key"
    assert kwargs["params"]["language"] == "en-US"
    assert kwargs["params"]["query"] == "inception"


def test_auth_error_raises_tmdb_error():
    session = Mock()
    session.get.return_value = FakeResponse(401)
    client = make_client(session)
    with pytest.raises(TmdbError, match="401"):
        client.search_movies("x")


def test_not_found_raises_tmdb_error():
    session = Mock()
    session.get.return_value = FakeResponse(404)
    client = make_client(session)
    with pytest.raises(TmdbError, match="404"):
        client.get_similar(27205)


def test_rate_limit_raises_tmdb_error():
    session = Mock()
    session.get.return_value = FakeResponse(429)
    client = make_client(session)
    with pytest.raises(TmdbError, match="429"):
        client.get_recommendations(27205)


def test_network_failure_raises_tmdb_error():
    session = Mock()
    session.get.side_effect = requests.ConnectionError("boom")
    client = make_client(session)
    with pytest.raises(TmdbError, match="request to"):
        client.search_movies("x")


def test_invalid_json_raises_tmdb_error():
    session = Mock()
    session.get.return_value = FakeResponse(200, ValueError("bad json"))
    client = make_client(session)
    with pytest.raises(TmdbError, match="invalid JSON"):
        client.search_movies("x")


# ---------- Sprint 2: response cache (store-backed) ----------


def test_cache_hit_skips_the_network():
    store = MovieStore(":memory:")
    session = Mock()
    session.get.return_value = FakeResponse(200, SEARCH_PAYLOAD)
    client = make_cached_client(session, store)
    first = client.search_movies("inception")
    second = client.search_movies("inception")
    assert session.get.call_count == 1
    assert second == first
    assert client.cache_misses == 1
    assert client.cache_hits == 1
    store.close()


def test_cache_key_ignores_the_api_key():
    store = MovieStore(":memory:")
    first_session = Mock()
    first_session.get.return_value = FakeResponse(200, SEARCH_PAYLOAD)
    make_cached_client(first_session, store).search_movies("inception")
    second_session = Mock()
    client = make_cached_client(second_session, store, api_key="other-key")
    results = client.search_movies("inception")
    assert second_session.get.call_count == 0
    assert [movie["id"] for movie in results] == [1, 2]
    store.close()


def test_errors_are_not_cached():
    store = MovieStore(":memory:")
    session = Mock()
    session.get.return_value = FakeResponse(401)
    client = make_cached_client(session, store)
    with pytest.raises(TmdbError, match="401"):
        client.search_movies("x")
    count = store.conn.execute("SELECT COUNT(*) FROM api_cache").fetchone()[0]
    assert count == 0
    store.close()


def test_corrupt_cache_entry_is_refetched():
    store = MovieStore(":memory:")
    session = Mock()
    session.get.return_value = FakeResponse(200, SEARCH_PAYLOAD)
    client = make_cached_client(session, store)
    client.search_movies("inception")
    store.conn.execute("UPDATE api_cache SET payload = 'not json'")
    store.conn.commit()
    results = client.search_movies("inception")
    assert session.get.call_count == 2
    assert [movie["id"] for movie in results] == [1, 2]
    payload = store.conn.execute("SELECT payload FROM api_cache").fetchone()[0]
    assert json.loads(payload)["results"][0]["id"] == 1
    store.close()


def test_expired_cache_entry_is_refetched():
    store = MovieStore(":memory:")
    session = Mock()
    session.get.return_value = FakeResponse(200, SEARCH_PAYLOAD)
    client = make_cached_client(session, store)
    client.search_movies("inception")
    store.conn.execute(
        "UPDATE api_cache SET fetched_at = datetime('now', '-2 days')"
    )
    store.conn.commit()
    client.search_movies("inception")
    assert session.get.call_count == 2
    store.close()


def test_client_without_store_does_not_cache():
    session = Mock()
    session.get.return_value = FakeResponse(200, SEARCH_PAYLOAD)
    client = make_client(session)
    client.search_movies("inception")
    client.search_movies("inception")
    assert session.get.call_count == 2
    assert client.cache_hits == 0
    assert client.cache_misses == 0


def test_get_movie_details_is_cached():
    store = MovieStore(":memory:")
    session = Mock()
    session.get.return_value = FakeResponse(
        200, {"id": 27205, "title": "Inception"}
    )
    client = make_cached_client(session, store)
    first = client.get_movie_details(27205)
    second = client.get_movie_details(27205)
    assert session.get.call_count == 1
    assert first == second == {"id": 27205, "title": "Inception"}
    store.close()


Writing tests/test_tmdb_client.py


In [31]:
%%writefile tests/test_movie_store.py
"""Unit tests for the SQLite persistence layer."""

import pytest

from src.movie_store import CACHE_TTL_SECONDS, MovieStore


@pytest.fixture()
def store(tmp_path):
    db = MovieStore(str(tmp_path / "test.db"))
    yield db
    db.close()


def movie(movie_id=101, title="Alpha Signal", rating=7.5):
    return {"id": movie_id, "title": title, "release_date": "2010-05-01",
            "vote_average": rating, "vote_count": 100, "popularity": 5.0}


def test_add_and_get_movie_roundtrip(store):
    store.add_movie(movie())
    stored = store.get_movie(101)
    assert stored["title"] == "Alpha Signal"
    assert stored["vote_average"] == 7.5


def test_upsert_refreshes_existing_movie(store):
    store.add_movie(movie(rating=7.5))
    store.add_movie(movie(rating=9.1))
    assert len(store.list_movies()) == 1
    assert store.get_movie(101)["vote_average"] == 9.1


def test_missing_movie_returns_none(store):
    assert store.get_movie(999) is None


def test_similar_link_roundtrip(store):
    store.add_similar_link(1, 101, score=8.0)
    store.add_similar_link(1, 102, score=6.0)
    links = store.get_similar_links(1)
    assert [link["target_id"] for link in links] == [101, 102]


def test_watchlist_add_list_and_flag(store):
    store.add_movie(movie())
    assert store.is_in_watchlist(101) is False
    store.add_to_watchlist(101, note="watch soon")
    assert store.is_in_watchlist(101) is True
    rows = store.get_watchlist()
    assert rows[0]["title"] == "Alpha Signal"
    assert rows[0]["note"] == "watch soon"


def test_store_creates_parent_directory(tmp_path):
    nested = tmp_path / "nested" / "dir" / "movies.db"
    db = MovieStore(str(nested))
    db.add_movie(movie())
    db.close()
    assert nested.exists()


# ---------- Sprint 2: watchlist removals, favorites, searches, cache ----------


def test_watchlist_remove_returns_true_then_false(store):
    store.add_movie(movie())
    store.add_to_watchlist(101)
    assert store.remove_from_watchlist(101) is True
    assert store.remove_from_watchlist(101) is False
    assert store.get_watchlist() == []


def test_watchlist_clear_reports_removed_count(store):
    store.add_movie(movie(101))
    store.add_movie(movie(102, title="Bravo Horizon"))
    store.add_to_watchlist(101)
    store.add_to_watchlist(102)
    assert store.clear_watchlist() == 2
    assert store.clear_watchlist() == 0
    assert store.get_watchlist() == []


def test_favorites_roundtrip_and_note_refresh(store):
    store.add_movie(movie())
    assert store.is_in_favorites(101) is False
    store.add_favorite(101, note="top pick")
    assert store.is_in_favorites(101) is True
    rows = store.get_favorites()
    assert rows[0]["title"] == "Alpha Signal"
    assert rows[0]["note"] == "top pick"
    store.add_favorite(101, note="changed")
    assert store.get_favorites()[0]["note"] == "changed"
    assert store.remove_favorite(101) is True
    assert store.remove_favorite(101) is False


def test_recent_searches_newest_first_with_limit(store):
    store.record_search("alpha", 3)
    store.record_search("bravo", 5)
    store.record_search("cobalt", 0)
    rows = store.get_recent_searches(2)
    assert [row["query"] for row in rows] == ["cobalt", "bravo"]
    assert rows[0]["result_count"] == 0
    assert rows[0]["searched_at"]
    assert len(store.get_recent_searches()) == 3


def test_cache_roundtrip_and_expiry(store):
    assert store.get_cached("k") is None
    store.save_cached("k", '{"a": 1}')
    assert store.get_cached("k") == '{"a": 1}'
    store.conn.execute(
        "UPDATE api_cache SET fetched_at = datetime('now', '-2 days') "
        "WHERE cache_key = 'k'"
    )
    store.conn.commit()
    assert store.get_cached("k") is None


def test_cache_save_overwrites_previous_payload(store):
    store.save_cached("k", "old")
    store.save_cached("k", "new")
    assert store.get_cached("k") == "new"


def test_cache_ttl_constant_is_24_hours():
    assert CACHE_TTL_SECONDS == 24 * 60 * 60


Writing tests/test_movie_store.py


In [32]:
%%writefile tests/test_discovery_flow.py
"""End-to-end logic tests: search, discovery filters, builder, export."""

import csv

import pytest

from fakes import FakeTmdbClient
from src.discovery import DiscoveryService
from src.exporter import LIST_COLUMNS, ExportError, Exporter
from src.movie_store import MovieStore
from src.sample_data import SAMPLE_MOVIES
from src.tmdb_client import TmdbError
from src.watchlist import WatchlistBuilder

SEED = 27205


@pytest.fixture()
def store(tmp_path):
    db = MovieStore(str(tmp_path / "s2.db"))
    yield db
    db.close()


def build_service(store, client=None):
    return DiscoveryService(client or FakeTmdbClient(), store)


def test_dedupe_merges_similar_and_recommendations(tmp_path):
    store = MovieStore(str(tmp_path / "a.db"))
    results = build_service(store).find_similar(SEED, limit=10)
    ids = [movie["id"] for movie in results]
    assert len(ids) == len(set(ids))  # movie 103 is in both source lists
    assert 103 in ids
    store.close()


def test_min_rating_filter(tmp_path):
    store = MovieStore(str(tmp_path / "b.db"))
    results = build_service(store).find_similar(SEED, min_rating=8.0)
    assert sorted(movie["id"] for movie in results) == [103, 105]
    store.close()


def test_year_range_excludes_unknown_dates(tmp_path):
    store = MovieStore(str(tmp_path / "c.db"))
    results = build_service(store).find_similar(SEED, year_from=2010,
                                                year_to=2020)
    assert sorted(movie["id"] for movie in results) == [101, 102]  # 104: no date
    store.close()


def test_ranking_prefers_rating_then_votes(tmp_path):
    store = MovieStore(str(tmp_path / "d.db"))
    results = build_service(store).find_similar(SEED)
    assert results[0]["id"] == 103  # 8.4 beats everything else
    store.close()


def test_limit_is_applied(tmp_path):
    store = MovieStore(str(tmp_path / "e.db"))
    results = build_service(store).find_similar(SEED, limit=2)
    assert len(results) == 2
    store.close()


def test_watchlist_builder_skips_saved_movies(tmp_path):
    store = MovieStore(str(tmp_path / "f.db"))
    results = build_service(store).find_similar(SEED)
    builder = WatchlistBuilder(store)
    first = builder.build(results, top_n=2)
    assert len(first) == 2
    again = builder.build(results, top_n=2)
    assert [movie["id"] for movie in again] == \
        [movie["id"] for movie in results[2:4]]
    store.close()


def test_csv_export_roundtrip(tmp_path):
    store = MovieStore(str(tmp_path / "g.db"))
    results = build_service(store).find_similar(SEED)
    WatchlistBuilder(store).build(results, top_n=3)
    out = tmp_path / "watchlist.csv"
    Exporter.export_watchlist(store, str(out))
    with open(out, newline="", encoding="utf-8") as handle:
        rows = list(csv.DictReader(handle))
    assert len(rows) == 3
    assert rows[0]["title"]
    store.close()


# ---------- Sprint 2: search, sorts, add-by-id, favorites, exports ----------


def test_search_caps_at_ten_and_logs_total(store):
    class ManyHits:
        def search_movies(self, query, page=1):
            return [{"id": i, "title": f"T{i}"} for i in range(1, 15)]

    results = DiscoveryService(ManyHits(), store).search("anything")
    assert len(results) == 10
    assert results[0]["title"] == "T1"
    rows = store.get_recent_searches(1)
    assert rows[0]["query"] == "anything"
    assert rows[0]["result_count"] == 14  # total, not just the 10 shown


def test_search_logs_the_query(store):
    client = FakeTmdbClient()
    results = build_service(store, client).search("alpha")
    assert [movie["title"] for movie in results] == ["Alpha Signal"]
    assert client.search_calls == ["alpha"]
    assert store.get_recent_searches(1)[0]["result_count"] == 1


def test_sort_votes_uses_vote_count_first(store):
    ids = [movie["id"] for movie in
           build_service(store).find_similar(SEED, sort="votes")]
    assert ids == [103, 101, 105, 102, 104]


def test_sort_year_puts_unknown_dates_last(store):
    ids = [movie["id"] for movie in
           build_service(store).find_similar(SEED, sort="year")]
    assert ids == [105, 102, 101, 103, 104]


def test_sort_popularity_descending(store):
    ids = [movie["id"] for movie in
           build_service(store).find_similar(SEED, sort="popularity")]
    assert ids == [103, 101, 105, 102, 104]  # 55, 40, 30, 22, 5


def test_unknown_sort_option_raises(store):
    with pytest.raises(ValueError, match="unknown sort"):
        build_service(store).find_similar(SEED, sort="imdb")


def test_only_kept_movies_are_saved_and_linked(store):
    results = build_service(store).find_similar(SEED, min_rating=8.0)
    assert sorted(movie["id"] for movie in results) == [103, 105]
    links = store.get_similar_links(SEED)
    assert [link["target_id"] for link in links] == [103, 105]  # score DESC
    assert store.get_movie(103) is not None
    assert store.get_movie(101) is None  # filtered out -> not stored


def test_ensure_movie_fetches_once_then_uses_store(store):
    client = FakeTmdbClient()
    builder = WatchlistBuilder(store, client)
    first = builder.ensure_movie(101)
    second = builder.ensure_movie(101)
    assert first["title"] == "Alpha Signal"
    assert second["title"] == "Alpha Signal"
    assert client.detail_calls == [101]


def test_add_returns_true_then_false(store):
    client = FakeTmdbClient()
    builder = WatchlistBuilder(store, client)
    assert builder.add(101) is True
    assert builder.add(101) is False
    assert store.is_in_watchlist(101) is True
    assert client.detail_calls == [101]


def test_add_favorite_refreshes_note(store):
    builder = WatchlistBuilder(store, FakeTmdbClient())
    assert builder.add_favorite(102, note="first") is True
    assert builder.add_favorite(102, note="second") is False
    rows = store.get_favorites()
    assert rows[0]["note"] == "second"


def test_ensure_movie_without_client_raises(store):
    builder = WatchlistBuilder(store)
    with pytest.raises(ValueError, match="no TMDB client"):
        builder.ensure_movie(101)


def test_ensure_movie_propagates_404(store):
    builder = WatchlistBuilder(store, FakeTmdbClient())
    with pytest.raises(TmdbError, match="404"):
        builder.ensure_movie(999)


def test_export_favorites_roundtrip(store, tmp_path):
    for entry in SAMPLE_MOVIES[:2]:
        store.add_movie(entry)
        store.add_favorite(entry["id"])
    out = tmp_path / "favorites.csv"
    assert Exporter.export_favorites(store, str(out)) == 2
    with open(out, newline="", encoding="utf-8") as handle:
        rows = list(csv.DictReader(handle))
    assert list(rows[0].keys()) == list(LIST_COLUMNS)
    assert {row["title"] for row in rows} == {"Alpha Signal",
                                              "Bravo Horizon"}


def test_export_keeps_tricky_titles_intact(store, tmp_path):
    tricky = {"id": 900, "title": 'Comma, "quoting" & ไทย',
              "release_date": "2024-01-01", "vote_average": 7.0,
              "vote_count": 10, "popularity": 1.0}
    store.add_movie(tricky)
    store.add_to_watchlist(900, note='note, with "quotes"')
    out = tmp_path / "tricky.csv"
    Exporter.export_watchlist(store, str(out))
    with open(out, newline="", encoding="utf-8") as handle:
        rows = list(csv.DictReader(handle))
    assert rows[0]["title"] == tricky["title"]
    assert rows[0]["note"] == 'note, with "quotes"'


def test_export_creates_nested_folders(store, tmp_path):
    store.add_movie(dict(SAMPLE_MOVIES[0]))
    store.add_to_watchlist(101)
    out = tmp_path / "deep" / "nested" / "watchlist.csv"
    assert Exporter.export_watchlist(store, str(out)) == 1
    assert out.exists()


def test_export_empty_list_writes_header_only(store, tmp_path):
    out = tmp_path / "empty.csv"
    assert Exporter.export_watchlist(store, str(out)) == 0
    text = out.read_text(encoding="utf-8")
    assert text.strip() == ",".join(LIST_COLUMNS)


def test_export_error_names_the_path(tmp_path):
    store = MovieStore(":memory:")
    blocker = tmp_path / "blocker"
    blocker.write_text("i am a file")
    with pytest.raises(ExportError, match="blocker"):
        Exporter.export_watchlist(store, str(blocker / "out.csv"))
    store.close()


Writing tests/test_discovery_flow.py


In [33]:
%%writefile tests/test_cli.py
"""Unit tests for the interactive CLI.

Inputs are scripted and the services are injected as offline fakes, so
every edge case from the PLAN.md Definition of Done is verified without a
real terminal, network, or database file.
"""

import pytest

from fakes import FakeTmdbClient
from src.cli import CLI
from src.movie_store import MovieStore
from src.sample_data import SAMPLE_MOVIES


def scripted_inputs(lines):
    """Return an input function that replays the given lines, then EOF."""
    queue = list(lines)

    def fake_input(prompt=""):
        if not queue:
            raise EOFError
        return queue.pop(0)

    return fake_input


def run_cli(lines, data_dir=None):
    """Run the CLI with scripted input and fake services; return (outputs, cli)."""
    outputs = []
    cli = CLI(input_func=scripted_inputs(lines), output_func=outputs.append,
              client=FakeTmdbClient(), store=MovieStore(":memory:"),
              data_dir=data_dir)
    cli.run()
    return outputs, cli


def joined(outputs):
    return "\n".join(outputs)


@pytest.mark.parametrize("command", ["quit", "QUIT", "Quit", "  quit  ",
                                     "exit", "q", "ออก"])
def test_quit_variants_exit_with_farewell(command):
    outputs, cli = run_cli([command])
    assert "ลาก่อน" in joined(outputs)
    assert cli.running is False


def test_eof_exits_gracefully():
    outputs, cli = run_cli([])
    assert "ลาก่อน" in joined(outputs)
    assert cli.running is False


def test_empty_input_warns_and_continues():
    outputs, _ = run_cli(["", "quit"])
    assert "กรุณาพิมพ์คำสั่ง" in joined(outputs)
    assert "ลาก่อน" in joined(outputs)


def test_unknown_command_warns_and_recovers():
    outputs, _ = run_cli(["asdf", "quit"])
    text = joined(outputs)
    assert "ไม่รู้จักคำสั่ง" in text
    assert "help" in text
    assert "ลาก่อน" in text


def test_search_requires_a_title():
    outputs, _ = run_cli(["search", "quit"])
    assert "ต้องระบุชื่อเรื่อง" in joined(outputs)


def test_search_finds_sample_movie():
    outputs, _ = run_cli(["search alpha", "quit"])
    assert "Alpha Signal" in joined(outputs)


def test_search_is_case_insensitive():
    outputs, _ = run_cli(["SEARCH ALPHA", "quit"])
    assert "Alpha Signal" in joined(outputs)


def test_discover_rejects_non_numeric_id():
    outputs, _ = run_cli(["discover abc", "quit"])
    assert "ต้องเป็นตัวเลข" in joined(outputs)


def test_discover_rejects_zero_and_negative():
    outputs, _ = run_cli(["discover 0", "discover -3", "quit"])
    assert joined(outputs).count("ต้องมากกว่า 0") == 2


def test_discover_unknown_seed_is_reported():
    outputs, _ = run_cli(["discover 999", "quit"])
    assert "ไม่พบรหัสภาพยนตร์ 999" in joined(outputs)


def test_watchlist_add_list_and_duplicate():
    outputs, cli = run_cli(
        ["watchlist add 101", "watchlist list", "watchlist add 101", "quit"]
    )
    text = joined(outputs)
    assert "เพิ่ม 'Alpha Signal'" in text
    assert "อยู่ในรายการรับชมแล้ว" in text
    assert len(cli.store.get_watchlist()) == 1


def test_watchlist_needs_subcommand():
    outputs, _ = run_cli(["watchlist", "quit"])
    assert "ต้องมีคำสั่งย่อย" in joined(outputs)


def test_help_shows_menu():
    outputs, _ = run_cli(["help", "quit"])
    text = joined(outputs)
    assert "search" in text
    assert "quit" in text


def test_prompt_marker_is_stable():
    prompts = []

    def capture(prompt=""):
        prompts.append(prompt)
        raise EOFError

    cli = CLI(input_func=capture, output_func=lambda *_: None,
              client=FakeTmdbClient(), store=MovieStore(":memory:"))
    cli.run()
    assert prompts and all(prompt == "movie> " for prompt in prompts)


# ---------- Sprint 2: favorites, export, history, flags, missing key ----------


def test_favorites_lifecycle():
    outputs, cli = run_cli(
        ["favorites add 101", "favorites add 101", "favorites list",
         "favorites remove 101", "favorites remove 101", "quit"]
    )
    text = joined(outputs)
    assert "เพิ่ม 'Alpha Signal' เข้ารายการโปรดแล้ว" in text
    assert text.count("อยู่ในรายการโปรดแล้ว") == 1
    assert "รายการโปรด (1 เรื่อง):" in text
    assert "นำ 'Alpha Signal' ออกจากรายการโปรดแล้ว" in text
    assert "ไม่พบรหัสภาพยนตร์ 101 ในรายการโปรด" in text
    assert cli.store.get_favorites() == []


def test_export_writes_csv_to_data_dir(tmp_path):
    outputs, _ = run_cli(
        ["watchlist add 101", "favorites add 102",
         "export watchlist", "export favorites", "quit"],
        data_dir=str(tmp_path)
    )
    text = joined(outputs)
    watch = tmp_path / "watchlist.csv"
    favorites = tmp_path / "favorites.csv"
    assert f"ส่งออก 1 รายการไปที่ {watch}" in text
    assert f"ส่งออก 1 รายการไปที่ {favorites}" in text
    assert "Alpha Signal" in watch.read_text(encoding="utf-8")


def test_export_requires_a_list_name():
    outputs, _ = run_cli(["export wrong", "export", "quit"])
    assert joined(outputs).count("ต้องระบุรายการ") == 2


def test_history_reports_recent_searches():
    outputs, _ = run_cli(["search alpha", "search bravo", "history",
                          "history 1", "quit"])
    text = joined(outputs)
    assert "ประวัติการค้นหาล่าสุด (2 รายการ):" in text
    assert "'alpha'" in text and "'bravo'" in text
    assert text.count("(1 รายการ):") == 1


def test_history_empty_message():
    outputs, _ = run_cli(["history", "quit"])
    assert "ยังไม่มีประวัติการค้นหา" in joined(outputs)


def test_watchlist_remove_and_clear():
    outputs, cli = run_cli(
        ["watchlist add 101", "watchlist remove 101", "watchlist remove 101",
         "watchlist clear", "quit"]
    )
    text = joined(outputs)
    assert "นำ 'Alpha Signal' ออกจากรายการรับชมแล้ว" in text
    assert "ไม่พบรหัสภาพยนตร์ 101 ในรายการรับชม" in text
    assert "ล้างรายการรับชมเรียบร้อยแล้ว (0 เรื่อง)" in text
    assert cli.store.get_watchlist() == []


@pytest.mark.parametrize("command,expected", [
    ("discover 101 --min-rating 11", "ต้องอยู่ระหว่าง 0 ถึง 10"),
    ("discover 101 --min-rating abc", "ต้องเป็นตัวเลข"),
    ("discover 101 --min-votes -5", "ต้องไม่ติดลบ"),
    ("discover 101 --min-votes x", "จำนวนเต็ม"),
    ("discover 101 --year 19x0", "4 หลัก"),
    ("discover 101 --year 2020-1990", "ช่วงปีต้องเรียง"),
    ("discover 101 --sort banana", "--sort ต้องเป็น"),
    ("discover 101 --limit 0", "ต้องมากกว่า 0"),
    ("discover 101 --limit x", "ต้องเป็นตัวเลข"),
    ("discover 101 --bogus 1", "ไม่รู้จักตัวเลือก"),
    ("discover 101 --limit", "ต้องมีค่าตามหลัง"),
])
def test_discover_option_validation(command, expected):
    outputs, _ = run_cli([command, "quit"])
    assert expected in joined(outputs)


def test_discover_sort_and_limit_applied():
    outputs, _ = run_cli(["discover 101 --sort votes --limit 2", "quit"])
    text = joined(outputs)
    assert "เรียงตาม votes" in text
    assert "[103]" in text and "[101]" in text and "[105]" not in text


def test_discover_year_sort_puts_unknown_last():
    outputs, _ = run_cli(["discover 101 --sort year", "quit"])
    text = joined(outputs)
    assert "1. [105]" in text and "5. [104]" in text


def test_missing_key_guides_and_keeps_offline_commands(
        monkeypatch, tmp_path):
    monkeypatch.delenv("TMDB_API_KEY", raising=False)
    outputs = []
    cli = CLI(input_func=scripted_inputs(
                  ["search alpha", "discover 101", "watchlist add 999",
                   "watchlist list", "history", "quit"]),
              output_func=outputs.append, store=MovieStore(":memory:"),
              data_dir=str(tmp_path))
    cli.run()
    text = joined(outputs)
    assert cli.key_missing is True
    assert text.count("ยังไม่ได้ตั้งค่าคีย์ TMDB") == 3
    assert "รายการรับชมยังว่างอยู่" in text
    assert "ยังไม่มีประวัติการค้นหา" in text


def test_offline_add_works_for_stored_movies(monkeypatch):
    monkeypatch.delenv("TMDB_API_KEY", raising=False)
    outputs = []
    store = MovieStore(":memory:")
    store.add_movie(dict(SAMPLE_MOVIES[0]))
    cli = CLI(input_func=scripted_inputs(["watchlist add 101",
                                          "watchlist list", "quit"]),
              output_func=outputs.append, store=store)
    cli.run()
    text = joined(outputs)
    assert "เพิ่ม 'Alpha Signal' เข้ารายการรับชมแล้ว" in text
    assert "[101] Alpha Signal" in text


def test_help_covers_sprint2_commands():
    outputs, _ = run_cli(["help", "quit"])
    text = joined(outputs)
    for command in ("favorites", "export", "history"):
        assert command in text


Writing tests/test_cli.py


In [34]:
%%writefile tests/test_web.py
"""Web-layer tests (Sprint 3): every route, offline, no key, no network.

Uses the Flask test client with the shared fakes and an in-memory store.
Everything runs offline: no key, no network, no generated files.
"""

import pytest

from fakes import FakeTmdbClient
from src.cli import CLI
from src.exporter import LIST_COLUMNS
from src.movie_store import MovieStore
from src.sample_data import SAMPLE_MOVIES
from src.tmdb_client import TmdbError
from src.watchlist import WatchlistBuilder
from web import create_app


class FlakyFake(FakeTmdbClient):
    """Fake client that raises TmdbError on the chosen calls."""

    def __init__(self, fail_on=(), message="HTTP 503 service unavailable"):
        super().__init__()
        self.fail_on = set(fail_on)
        self.message = message

    def search_movies(self, query, page=1):
        if "search" in self.fail_on:
            raise TmdbError(self.message)
        return super().search_movies(query, page)

    def get_movie_details(self, tmdb_id):
        if "detail" in self.fail_on:
            raise TmdbError(self.message)
        return super().get_movie_details(tmdb_id)

    def get_similar(self, tmdb_id, page=1):
        if "similar" in self.fail_on:
            raise TmdbError(self.message)
        return super().get_similar(tmdb_id, page)

    def get_recommendations(self, tmdb_id, page=1):
        if "similar" in self.fail_on:
            raise TmdbError(self.message)
        return super().get_recommendations(tmdb_id, page)


@pytest.fixture()
def store():
    """Fresh in-memory store for one test."""
    instance = MovieStore(":memory:")
    yield instance
    instance.close()


def make_client(tmp_path, store, client):
    """App with ``client`` (None = keyless mode) and a temp data dir."""
    app = create_app(client=client, store=store, data_dir=str(tmp_path))
    return app.test_client()


class TestHome:
    def test_home_opens_with_key(self, tmp_path, store):
        resp = make_client(tmp_path, store, FakeTmdbClient()).get("/")
        assert resp.status_code == 200
        assert "Movie Discovery" in resp.get_data(as_text=True)

    def test_home_shows_key_banner_when_missing(self, tmp_path, store):
        resp = make_client(tmp_path, store, None).get("/")
        assert resp.status_code == 200
        assert "ยังไม่ได้ตั้งค่าคีย์" in resp.get_data(as_text=True)


class TestSearch:
    def test_results_page(self, tmp_path, store):
        client = make_client(tmp_path, store, FakeTmdbClient())
        needle = SAMPLE_MOVIES[0]["title"]
        resp = client.get(f"/search?q={needle}")
        assert resp.status_code == 200
        assert needle in resp.get_data(as_text=True)

    def test_empty_query_gets_hint(self, tmp_path, store):
        resp = make_client(tmp_path, store, FakeTmdbClient()).get("/search?q=")
        assert "กรุณาพิมพ์คำค้นก่อนกดค้นหา" in resp.get_data(as_text=True)

    def test_whitespace_query_gets_hint(self, tmp_path, store):
        resp = make_client(tmp_path, store, FakeTmdbClient()).get(
            "/search?q=%20%20%20")
        assert "กรุณาพิมพ์คำค้นก่อนกดค้นหา" in resp.get_data(as_text=True)

    def test_keyless_guidance(self, tmp_path, store):
        resp = make_client(tmp_path, store, None).get("/search?q=test")
        html = resp.get_data(as_text=True)
        assert resp.status_code == 200
        assert "TMDB_API_KEY" in html

    def test_no_results_message(self, tmp_path, store):
        resp = make_client(tmp_path, store, FakeTmdbClient()).get(
            "/search?q=zzz-nothing")
        assert "ไม่พบภาพยนตร์ที่ตรงกับ" in resp.get_data(as_text=True)

    def test_long_query_truncated(self, tmp_path, store):
        client = make_client(tmp_path, store, FakeTmdbClient())
        resp = client.get("/search?q=" + "a" * 300)
        html = resp.get_data(as_text=True)
        assert resp.status_code == 200
        assert "ยาวเกิน 120" in html

    def test_emoji_query_does_not_crash(self, tmp_path, store):
        client = make_client(tmp_path, store, FakeTmdbClient())
        resp = client.get("/search?q=%F0%9F%98%80%F0%9F%91%8D")
        assert resp.status_code == 200

    def test_script_query_is_escaped(self, tmp_path, store):
        client = make_client(tmp_path, store, FakeTmdbClient())
        resp = client.get("/search?q=%3Cscript%3Ealert(1)%3C%2Fscript%3E")
        html = resp.get_data(as_text=True)
        assert "<script>alert(1)</script>" not in html
        assert "&lt;script&gt;" in html

    @pytest.mark.parametrize("message", [
        "HTTP 401 unauthorized",
        "HTTP 429 too many requests",
        "HTTP 503 service unavailable",
    ])
    def test_tmdb_errors_use_cli_wording(self, tmp_path, message):
        app = create_app(client=FlakyFake(fail_on=("search",),
                                          message=message),
                         store=MovieStore(":memory:"),
                         data_dir=str(tmp_path))
        resp = app.test_client().get("/search?q=x")
        assert resp.status_code == 502
        expected = CLI.describe_tmdb_error(TmdbError(message))
        assert expected in resp.get_data(as_text=True)


class TestMovieDetail:
    def test_detail_page_with_buttons(self, tmp_path, store):
        client = make_client(tmp_path, store, FakeTmdbClient())
        movie = SAMPLE_MOVIES[0]
        resp = client.get(f"/movie/{movie['id']}")
        html = resp.get_data(as_text=True)
        assert resp.status_code == 200
        assert movie["title"] in html
        assert f"/watchlist/add/{movie['id']}" in html
        assert f"/favorites/add/{movie['id']}" in html

    def test_bad_filter_values_fall_back_with_notes(self, tmp_path, store):
        client = make_client(tmp_path, store, FakeTmdbClient())
        mid = SAMPLE_MOVIES[0]["id"]
        resp = client.get(
            f"/movie/{mid}?min_rating=abc&limit=999"
            "&sort=nope&year_from=2000&year_to=1990")
        html = resp.get_data(as_text=True)
        assert resp.status_code == 200
        assert "ไม่ใช่ตัวเลข" in html
        assert "อยู่นอกช่วงที่รองรับ" in html
        assert "ไม่รู้จักวิธีเรียง" in html
        assert "ช่วงปีสลับกัน" in html

    def test_unknown_movie_id_polite_404(self, tmp_path, store):
        client = make_client(tmp_path, store, FakeTmdbClient())
        resp = client.get("/movie/424242")
        html = resp.get_data(as_text=True)
        assert resp.status_code == 404
        assert "ไม่พบภาพยนตร์" in html

    @pytest.mark.parametrize("bad_path", ["/movie/abc", "/movie/-5"])
    def test_non_matching_ids_404(self, tmp_path, store, bad_path):
        client = make_client(tmp_path, store, FakeTmdbClient())
        assert client.get(bad_path).status_code == 404

    def test_similar_failure_shows_inline(self, tmp_path, store):
        app = create_app(client=FlakyFake(fail_on=("similar",),
                                          message="HTTP 503 sim failed"),
                         store=store, data_dir=str(tmp_path))
        resp = app.test_client().get(f"/movie/{SAMPLE_MOVIES[0]['id']}")
        html = resp.get_data(as_text=True)
        assert resp.status_code == 200
        assert SAMPLE_MOVIES[0]["title"] in html
        expected = CLI.describe_tmdb_error(TmdbError("HTTP 503 sim failed"))
        assert expected in html

    def test_detail_down_returns_502(self, tmp_path, store):
        app = create_app(client=FlakyFake(fail_on=("detail",),
                                          message="HTTP 503 detail down"),
                         store=store, data_dir=str(tmp_path))
        resp = app.test_client().get(f"/movie/{SAMPLE_MOVIES[0]['id']}")
        assert resp.status_code == 502

    def test_offline_mode_saved_movie(self, tmp_path, store):
        store.add_movie(dict(SAMPLE_MOVIES[0]))
        resp = make_client(tmp_path, store, None).get(
            f"/movie/{SAMPLE_MOVIES[0]['id']}")
        html = resp.get_data(as_text=True)
        assert resp.status_code == 200
        assert "โหมดไม่มีคีย์" in html

    def test_offline_mode_unsaved_movie_404(self, tmp_path, store):
        resp = make_client(tmp_path, store, None).get("/movie/424242")
        html = resp.get_data(as_text=True)
        assert resp.status_code == 404
        assert "ไม่มีข้อมูลภาพยนตร์เรื่องนี้ในเครื่อง" in html


class TestListActions:
    def test_watchlist_add_exists_remove_missing(self, tmp_path, store):
        client = make_client(tmp_path, store, FakeTmdbClient())
        mid = SAMPLE_MOVIES[1]["id"]

        def text(resp):
            return resp.get_data(as_text=True)

        resp = client.post(f"/watchlist/add/{mid}", follow_redirects=True)
        assert "เพิ่มเข้าสู่รายการรับชมแล้ว" in text(resp)
        resp = client.post(f"/watchlist/add/{mid}", follow_redirects=True)
        assert "อยู่ในรายการรับชมอยู่แล้ว" in text(resp)
        resp = client.post(f"/watchlist/remove/{mid}", follow_redirects=True)
        assert "ลบออกจากรายการรับชมแล้ว" in text(resp)
        resp = client.post(f"/watchlist/remove/{mid}", follow_redirects=True)
        assert "ไม่พบเรื่องนี้ในรายการรับชม" in text(resp)

    def test_favorites_add_and_duplicate(self, tmp_path, store):
        client = make_client(tmp_path, store, FakeTmdbClient())
        mid = SAMPLE_MOVIES[0]["id"]
        resp = client.post(f"/favorites/add/{mid}", follow_redirects=True)
        assert "เพิ่มเข้าสู่รายการโปรดแล้ว" in resp.get_data(as_text=True)
        resp = client.post(f"/favorites/add/{mid}", follow_redirects=True)
        assert "อยู่ในรายการโปรดอยู่แล้ว" in resp.get_data(as_text=True)

    @pytest.mark.parametrize("evil", [
        "//evil.example/x",
        "/\\evil.example/x",
        "http://evil.example/x",
        "",
    ])
    def test_next_is_restricted_to_local_paths(self, tmp_path, store, evil):
        client = make_client(tmp_path, store, FakeTmdbClient())
        mid = SAMPLE_MOVIES[0]["id"]
        resp = client.post(f"/watchlist/add/{mid}", data={"next": evil})
        assert resp.status_code == 303
        assert resp.headers["Location"].startswith(f"/movie/{mid}")

    def test_clear_watchlist_flow(self, tmp_path, store):
        client = make_client(tmp_path, store, FakeTmdbClient())
        mid = SAMPLE_MOVIES[1]["id"]
        client.post(f"/watchlist/add/{mid}")
        resp = client.post("/watchlist/clear", follow_redirects=True)
        html = resp.get_data(as_text=True)
        assert "ล้างรายการรับชมเรียบร้อยแล้ว (1 เรื่อง)" in html
        assert "ยังไม่มีรายการ" in html

    def test_keyless_add_without_local_data(self, tmp_path, store):
        client = make_client(tmp_path, store, None)
        resp = client.post("/watchlist/add/424242")
        assert resp.status_code == 303
        assert "error=nodata" in resp.headers["Location"]


class TestListPages:
    def test_watchlist_page_shows_rows(self, tmp_path, store):
        WatchlistBuilder(store, FakeTmdbClient()).add(SAMPLE_MOVIES[0]["id"])
        resp = make_client(tmp_path, store, FakeTmdbClient()).get("/watchlist")
        html = resp.get_data(as_text=True)
        assert resp.status_code == 200
        assert "รายการรับชม (1)" in html
        assert SAMPLE_MOVIES[0]["title"] in html

    def test_empty_list_notice(self, tmp_path, store):
        resp = make_client(tmp_path, store, FakeTmdbClient()).get("/favorites")
        assert "ยังไม่มีรายการ" in resp.get_data(as_text=True)

    def test_cross_list_star_only_when_shared(self, tmp_path, store):
        shared = SAMPLE_MOVIES[0]["id"]
        only_watch = SAMPLE_MOVIES[1]["id"]
        builder = WatchlistBuilder(store, FakeTmdbClient())
        builder.add(shared)
        builder.add(only_watch)
        builder.add_favorite(shared)
        client = make_client(tmp_path, store, FakeTmdbClient())
        html_watch = client.get("/watchlist").get_data(as_text=True)
        html_fav = client.get("/favorites").get_data(as_text=True)
        assert html_watch.count('class="star"') == 1
        assert html_fav.count('class="star"') == 1

    def test_history_lists_searches(self, tmp_path, store):
        client = make_client(tmp_path, store, FakeTmdbClient())
        needle = SAMPLE_MOVIES[0]["title"]
        client.get(f"/search?q={needle}")
        resp = client.get("/history")
        html = resp.get_data(as_text=True)
        assert resp.status_code == 200
        assert needle in html

    def test_history_bad_limit_falls_back(self, tmp_path, store):
        client = make_client(tmp_path, store, FakeTmdbClient())
        assert client.get("/history?limit=abc").status_code == 200
        assert client.get("/history?limit=999").status_code == 200

    def test_export_watchlist_csv_content(self, tmp_path, store):
        WatchlistBuilder(store, FakeTmdbClient()).add(SAMPLE_MOVIES[0]["id"])
        client = make_client(tmp_path, store, FakeTmdbClient())
        resp = client.get("/export/watchlist")
        assert resp.status_code == 200
        assert "attachment" in resp.headers["Content-Disposition"]
        body = resp.get_data().decode("utf-8-sig")
        lines = body.splitlines()
        assert lines[0] == ",".join(LIST_COLUMNS)
        assert str(SAMPLE_MOVIES[0]["id"]) in lines[1]

    def test_export_empty_is_header_only(self, tmp_path, store):
        client = make_client(tmp_path, store, FakeTmdbClient())
        resp = client.get("/export/favorites")
        body = resp.get_data().decode("utf-8-sig")
        assert resp.status_code == 200
        assert body.splitlines() == [",".join(LIST_COLUMNS)]

    def test_export_bad_kind_404(self, tmp_path, store):
        client = make_client(tmp_path, store, FakeTmdbClient())
        assert client.get("/export/whatever").status_code == 404


class TestErrorPages:
    def test_unknown_url_404(self, tmp_path, store):
        resp = make_client(tmp_path, store, FakeTmdbClient()).get("/no-such")
        assert resp.status_code == 404
        assert "ไม่พบหน้าที่ต้องการ" in resp.get_data(as_text=True)

    def test_corrupt_database_shows_repair_page(self, tmp_path):
        bad_db = tmp_path / "movies.db"
        bad_db.write_bytes(b"this is definitely not a sqlite database " * 10)
        app = create_app(db_path=str(bad_db), data_dir=str(tmp_path))
        client = app.test_client()
        resp = client.get("/")
        html = resp.get_data(as_text=True)
        assert resp.status_code == 500
        assert "ฐานข้อมูลมีปัญหา" in html
        assert "ย้ายหรือลบ" in html
        assert "movies.db" in html
        assert client.get("/watchlist").status_code == 500


Writing tests/test_web.py


### 3.1 สาธิตการทำงานและกรณีขอบเขต (ผลลัพธ์จริงจากการรัน)

เซลล์ถัดไปรันเว็บแอปจริงผ่าน Flask test client และรัน CLI จริงโดยจำลองอินพุตผู้ใช้ (ไม่ใช่ภาพประกอบ) —
ทุกอย่างออฟไลน์ 100% และผลลัพธ์ที่เห็นคือผลจากการรันจริงในครั้งนี้


In [35]:
# สาธิตเว็บแอปแบบออฟไลน์ผ่าน Flask test client (ตัวจำลอง TMDB + ฐานข้อมูลชั่วคราว)
import pathlib
import tempfile

from fakes import FakeTmdbClient
from src.movie_store import MovieStore
from web import create_app


def check(label, resp, needles):
    """แสดงสถานะ HTTP และรายการข้อความที่ต้องพบในหน้า"""
    html = resp.get_data(as_text=True)
    print(f"[{label}] HTTP {resp.status_code} | {len(html)} ตัวอักษร")
    for needle in needles:
        print("   -", ("พบ" if needle in html else "ไม่พบ"), "->", needle)


work = pathlib.Path(tempfile.mkdtemp(prefix="sprint3-web-"))
app = create_app(client=FakeTmdbClient(), store=MovieStore(":memory:"),
                 data_dir=str(work))
c = app.test_client()

check("หน้าแรก", c.get("/"), ["ระบบค้นพบภาพยนตร์", "รายการรับชม (0)"])
check("ค้นหา alpha", c.get("/search?q=alpha"), ["Alpha Signal"])
check("รายละเอียดหนัง 101", c.get("/movie/101"),
      ["Alpha Signal", "Cobalt Night"])

r = c.post("/watchlist/add/101", follow_redirects=True)
check("เพิ่ม 101 เข้า watchlist", r,
      ["เพิ่มเข้าสู่รายการรับชมแล้ว", "รายการรับชม (1)"])
r = c.post("/watchlist/add/101", follow_redirects=True)
check("เพิ่มซ้ำ (ต้องไม่เพิ่ม)", r, ["เรื่องนี้อยู่ในรายการรับชมอยู่แล้ว"])
r = c.post("/watchlist/add/103", follow_redirects=True)
check("เพิ่ม 103", r, ["รายการรับชม (2)"])
r = c.post("/favorites/add/103", follow_redirects=True)
check("เพิ่ม 103 เข้า favorites", r, ["เพิ่มเข้าสู่รายการโปรดแล้ว"])
check("หน้า watchlist (ป้าย ★ ไขว้ลิสต์)", c.get("/watchlist"),
      ["Cobalt Night", "★", "รายการรับชม (2)"])
check("หน้า favorites", c.get("/favorites"), ["Cobalt Night"])
check("ประวัติการค้นหา", c.get("/history"), ["alpha"])

r = c.get("/export/watchlist")
print(f"[ดาวน์โหลด watchlist.csv] HTTP {r.status_code} | {len(r.data)} ไบต์")
print("   ตัวอย่างเนื้อหา:", r.data.decode("utf-8").splitlines()[:2])


[หน้าแรก] HTTP 200 | 1308 ตัวอักษร
   - พบ -> ระบบค้นพบภาพยนตร์
   - พบ -> รายการรับชม (0)
[ค้นหา alpha] HTTP 200 | 1366 ตัวอักษร
   - พบ -> Alpha Signal
[รายละเอียดหนัง 101] HTTP 200 | 4283 ตัวอักษร
   - พบ -> Alpha Signal
   - พบ -> Cobalt Night
[เพิ่ม 101 เข้า watchlist] HTTP 200 | 4446 ตัวอักษร
   - พบ -> เพิ่มเข้าสู่รายการรับชมแล้ว
   - พบ -> รายการรับชม (1)
[เพิ่มซ้ำ (ต้องไม่เพิ่ม)] HTTP 200 | 4453 ตัวอักษร
   - พบ -> เรื่องนี้อยู่ในรายการรับชมอยู่แล้ว
[เพิ่ม 103] HTTP 200 | 4446 ตัวอักษร
   - พบ -> รายการรับชม (2)
[เพิ่ม 103 เข้า favorites] HTTP 200 | 4501 ตัวอักษร
   - พบ -> เพิ่มเข้าสู่รายการโปรดแล้ว
[หน้า watchlist (ป้าย ★ ไขว้ลิสต์)] HTTP 200 | 2190 ตัวอักษร
   - พบ -> Cobalt Night
   - พบ -> ★
   - พบ -> รายการรับชม (2)
[หน้า favorites] HTTP 200 | 1533 ตัวอักษร
   - พบ -> Cobalt Night
[ประวัติการค้นหา] HTTP 200 | 1410 ตัวอักษร
   - พบ -> alpha
[ดาวน์โหลด watchlist.csv] HTTP 200 | 114 ไบต์
   ตัวอย่างเนื้อหา: ['tmdb_id,title,release_date,vote_average,note', '101,Alpha Signal

In [36]:
# กรณีขอบเขตและความปลอดภัยของเว็บ
from src.tmdb_client import TmdbError


class FlakyFake(FakeTmdbClient):
    """ตัวจำลองที่การค้นหาล้มเสมอ (จำลองสถานการณ์ TMDB ล่ม)"""

    def search_movies(self, query, page=1):
        raise TmdbError("HTTP 503 service unavailable")


check("URL ไม่มีอยู่", c.get("/nope"), ["ไม่พบหน้าที่ต้องการ"])

app2 = create_app(client=FlakyFake(), store=MovieStore(":memory:"),
                  data_dir=str(work))
check("TMDB ล่ม (502)", app2.test_client().get("/search?q=alpha"),
      ["เชื่อมต่อ TMDB ไม่สำเร็จ"])

bad = chr(92) * 2 + "evil.example"
r = c.post("/watchlist/add/101", data={"next": bad})
print("[open redirect: next แบบ backslash] HTTP", r.status_code,
      "| Location:", r.headers.get("Location"))
r = c.post("/watchlist/add/101", data={"next": "//evil.example"})
print("[open redirect: next แบบ //] HTTP", r.status_code,
      "| Location:", r.headers.get("Location"))

app3 = create_app(client=None, store=MovieStore(":memory:"),
                  data_dir=str(work))
c3 = app3.test_client()
check("โหมดไม่มีคีย์ — หน้าแรก", c3.get("/"), ["ยังไม่ได้ตั้งค่าคีย์"])
check("โหมดไม่มีคีย์ — ค้นหา", c3.get("/search?q=x"), ["TMDB_API_KEY"])
r = c3.get("/watchlist")
print(f"[โหมดไม่มีคีย์ — watchlist ยังใช้ได้] HTTP {r.status_code}")


[URL ไม่มีอยู่] HTTP 404 | 913 ตัวอักษร
   - พบ -> ไม่พบหน้าที่ต้องการ
[TMDB ล่ม (502)] HTTP 502 | 919 ตัวอักษร
   - พบ -> เชื่อมต่อ TMDB ไม่สำเร็จ
[open redirect: next แบบ backslash] HTTP 303 | Location: /movie/101?exists=watchlist
[open redirect: next แบบ //] HTTP 303 | Location: /movie/101?exists=watchlist
[โหมดไม่มีคีย์ — หน้าแรก] HTTP 200 | 1548 ตัวอักษร
   - พบ -> ยังไม่ได้ตั้งค่าคีย์
[โหมดไม่มีคีย์ — ค้นหา] HTTP 200 | 1379 ตัวอักษร
   - พบ -> TMDB_API_KEY
[โหมดไม่มีคีย์ — watchlist ยังใช้ได้] HTTP 200


In [37]:
# CLI เดิมยังทำงาน และใช้ฐานข้อมูลชุดเดียวกับเว็บ
from src.cli import CLI


def scripted(lines):
    """ฟังก์ชันอินพุตจำลอง: คืนค่าทีละบรรทัด แล้วส่ง EOF เมื่ออินพุตหมด"""
    queue = list(lines)

    def feed(prompt=""):
        if not queue:
            raise EOFError
        value = queue.pop(0)
        print(prompt + value)
        return value

    return feed


work5 = pathlib.Path(tempfile.mkdtemp(prefix="sprint3-shared-"))
db5 = str(work5 / "movies.db")
store5 = MovieStore(db5)
print("--- CLI เซสชัน: เพิ่มรายการลงฐานข้อมูลจริง ---")
CLI(input_func=scripted(["watchlist add 101", "quit"]), output_func=print,
    client=FakeTmdbClient(), store=store5, data_dir=str(work5)).run()
store5.close()

store6 = MovieStore(db5)
app5 = create_app(client=FakeTmdbClient(), store=store6, data_dir=str(work5))
check("เว็บอ่านฐานข้อมูลชุดเดียวกับ CLI", app5.test_client().get("/watchlist"),
      ["Alpha Signal", "รายการรับชม (1)"])


--- CLI เซสชัน: เพิ่มรายการลงฐานข้อมูลจริง ---
  Movie Discovery & Watchlist Builder — CLI
  ระบบค้นพบภาพยนตร์และสร้างรายการรับชม
คำสั่งที่ใช้ได้:
  search <ชื่อเรื่อง>              ค้นหาภาพยนตร์จาก TMDB
  discover <รหัส> [ตัวเลือก]       ภาพยนตร์ที่คล้ายกัน (กรอง/เรียงได้)
        --min-rating 0-10 · --min-votes N · --year YYYY|YYYY-YYYY
        --sort rating|votes|year|popularity · --limit N
  watchlist add <รหัส>            เพิ่มเข้ารายการรับชม
  watchlist list                  แสดงรายการรับชม
  watchlist remove <รหัส>         นำออกจากรายการรับชม
  watchlist clear                 ล้างรายการรับชมทั้งหมด
  favorites add <รหัส>            เพิ่มเข้ารายการโปรด
  favorites list                  แสดงรายการโปรด
  favorites remove <รหัส>         นำออกจากรายการโปรด
  export watchlist|favorites      ส่งออกรายการเป็น CSV (โฟลเดอร์ data/)
  history [จำนวน]                 ประวัติการค้นหาล่าสุด (ค่าเริ่มต้น 10)
  help                            แสดงคำสั่งทั้งหมด
  quit / exit / ออก               

In [38]:
# รันชุดทดสอบอัตโนมัติทั้ง 133 เคส (ออฟไลน์ 100% ไม่ต้องใช้คีย์จริง)
import sys
!{sys.executable} -m pytest -q


........................................................................ [ 54%]
.............................................................            [100%]
133 passed in 1.93s


In [39]:
# ตรวจสอบมาตรฐานโค้ด (PEP 8) ทั้งโปรเจกต์ รวมเว็บ
import sys
!{sys.executable} -m flake8 src tests web
print("ผลตรวจ flake8 ด้านบน (ว่าง = ผ่านทั้งหมด)")


ผลตรวจ flake8 ด้านบน (ว่าง = ผ่านทั้งหมด)


In [40]:
# ไฟล์ทั้งหมดที่สร้างขึ้นจากรายงานนี้
import pathlib

skip = {"__pycache__", ".pytest_cache", ".ipynb_checkpoints"}
for path in sorted(pathlib.Path(".").rglob("*")):
    if path.is_dir():
        continue
    if any(part in skip for part in path.parts):
        continue
    print("  " + path.relative_to(".").as_posix())


  .env.example
  requirements.txt
  setup.cfg
  src/__init__.py
  src/app.py
  src/cli.py
  src/config.py
  src/discovery.py
  src/exporter.py
  src/movie_store.py
  src/sample_data.py
  src/tmdb_client.py
  src/watchlist.py
  tests/conftest.py
  tests/fakes.py
  tests/test_cli.py
  tests/test_discovery_flow.py
  tests/test_movie_store.py
  tests/test_tmdb_client.py
  tests/test_web.py
  web/__init__.py
  web/__main__.py
  web/routes.py
  web/static/style.css
  web/templates/_card.html
  web/templates/base.html
  web/templates/error.html
  web/templates/history.html
  web/templates/home.html
  web/templates/list.html
  web/templates/movie.html
  web/templates/search.html


## ส่วนที่ 4: ผลลัพธ์และการทดสอบ (Result)

### 4.1 สรุปความก้าวหน้าของงาน (Sprint Progress Summary)

- [x] เว็บแอปครบทุกหน้าจอตามข้อ 2.3 — ใช้บริการและฐานข้อมูลชุดเดียวกับ CLI (พิสูจน์ในเซลล์ "CLI + ฐานข้อมูลชุดเดียว")
- [x] ป้าย ★ ไขว้ลิสต์ · จำนวนรายการบนแถบนำทางอัปเดตทันที · ไฟล์ CSV ตรงกับลิสต์ปัจจุบันเสมอ
- [x] ชุดทดสอบ 133 เคส ผ่านครบทุกเคส — ชุดเดิม 93 เคสไม่ถูกแก้ไขแม้แต่ไฟล์เดียว + ใหม่ 40 เคสสำหรับเว็บ (ดูผลในเซลล์ pytest ด้านบน)
- [x] flake8 ผ่านทั้งโปรเจกต์ รวม `web/`
- [x] เอกสารประกอบรอบนี้: `sprint3/PLAN.md`
- [ ] ส่งมอบงานผ่าน Pull Request บน GitHub (จะเปิด PR เมื่อถึงขั้นส่งมอบตามข้อตกลงของทีม)

### 4.2 ผลการทดสอบกรณีขอบเขต (QA Report)

| รายการทดสอบ | อินพุต | ผลลัพธ์ที่คาดหวัง | ผลการทดสอบจริง | สถานะ |
|---|---|---|---|---|
| หน้าแรก + แถบนำทาง | `GET /` | เปิดได้ พร้อมจำนวนรายการจากฐานข้อมูล | HTTP 200 — แสดง "รายการรับชม (0)" | PASSED |
| ค้นหาบนเว็บ | `GET /search?q=alpha` | แสดงผลการค้นหา | HTTP 200 — พบ Alpha Signal | PASSED |
| รายละเอียด + หนังคล้าย | `GET /movie/101` | แสดงข้อมูลหนังและรายการคล้ายกัน | HTTP 200 — พบ Alpha Signal + Cobalt Night | PASSED |
| เพิ่มเข้า watchlist (เว็บ) | `POST /watchlist/add/101` | แจ้งผล + ยอดบนแถบนำทางอัปเดต | "เพิ่มเข้าสู่รายการรับชมแล้ว" · ยอด (1) | PASSED |
| เพิ่มซ้ำต้องไม่เพิ่ม | `POST` ซ้ำ | แจ้งว่ามีอยู่แล้ว | "เรื่องนี้อยู่ในรายการรับชมอยู่แล้ว" | PASSED |
| ป้าย ★ ไขว้ลิสต์ | หนังอยู่ทั้งสองลิสต์ | หน้า watchlist ขึ้น ★ | พบ ★ ข้างรายการ Cobalt Night | PASSED |
| ดาวน์โหลด CSV | `GET /export/watchlist` | ได้ไฟล์หัวตาราง + ข้อมูลตรงลิสต์ | HTTP 200 · attachment · หัวตาราง + 2 แถว (114 ไบต์) | PASSED |
| URL ไม่พบ | `/nope` | 404 แบบสุภาพ | HTTP 404 | PASSED |
| TMDB ล่ม | ค้นหาขณะ TMDB ผิดพลาด | หน้า 502 แบบสุภาพ | HTTP 502 — "เชื่อมต่อ TMDB ไม่สำเร็จ" | PASSED |
| ป้องกัน open redirect | `next` เป็น URL ภายนอก (แบบ backslash และ //) | ไม่พาออกจากแอป | Location ยังคงเป็นปลายทางภายใน (/movie/101?exists=watchlist) | PASSED |
| โหมดไม่มีคีย์ | เปิดทุกหน้าโดยไม่ตั้ง `TMDB_API_KEY` | เปิดได้ + แบนเนอร์วิธีตั้งค่า | แบนเนอร์แสดง · watchlist ใช้งานได้ (HTTP 200) | PASSED |
| CLI + เว็บ ฐานข้อมูลชุดเดียว | CLI เพิ่มรายการ → เปิดเว็บ | เห็นข้อมูลเดียวกัน | เว็บแสดง Alpha Signal · ยอด (1) | PASSED |

### 4.3 สรุปบทเรียน (Retrospective: Wow! & Whoops!)

**Wow! (ส่วนที่ทำได้ดี):**

- ไม่แตะโค้ดเดิม: `src/` ทั้ง 10 โมดูลเหมือนรอบก่อนทุกไบต์ — เว็บเชื่อมผ่านบริการเดิมล้วน ๆ
- ชุดทดสอบ 133 เคส ผ่านครบ + flake8 ทั้งโปรเจกต์ (รวม web/) — ชุดเดิม 93 เคสผ่านโดยไม่แก้ไฟล์
- ขอบเขตครบ: คีย์หาย · TMDB ล่ม · 404 · ฐานข้อมูลเสีย — ทุกกรณีมีหน้าและข้อความแบบสุภาพ

**Whoops! (ปัญหาและแนวทางแก้ไข):**

- SQLite ใช้ข้าม thread ไม่ได้ (เซิร์ฟเวอร์เว็บจัดการคำขอใน worker thread) — แก้โดยเปิด–ปิดการเชื่อมต่อใหม่ต่อหนึ่งคำขอ
- พบช่อง open redirect: ค่า next รูปแบบ backslash หลุดการตรวจ — แก้โดยปฏิเสธทั้ง backslash และ // พร้อมเทสต์ถาวรกันกลับมา (เซลล์ด้านบนแสดงให้เห็นว่าถูกบล็อกจริง)

### 4.4 ลิงก์และขั้นตอนถัดไป

- **Repository:** https://github.com/somjung/Movie-Discovery
- **เอกสารประกอบ:** `sprint3/PLAN.md` (แผนงาน + ตารางกรณีขอบเขต)
- **Sprint ถัดไป (Sprint สุดท้าย):** DevOps · CI/CD และการผสาน AI ตามคู่มือรายวิชา — ต่อยอดจากระบบที่สมบูรณ์รอบนี้


---

*หมายเหตุ: เซลล์สาธิตทั้งหมดในรายงานนี้รันแบบออฟไลน์ 100% — ใช้ตัวจำลอง TMDB ชุดเดียวกับชุดทดสอบ (`tests/fakes.py`) และฐานข้อมูลชั่วคราว จึงไม่ต้องใช้คีย์หรืออินเทอร์เน็ต และรันซ้ำได้ทุกเครื่อง · โน้ตบุ๊กนี้สร้างจากไฟล์โค้ดจริงในโปรเจกต์ด้วยสคริปต์ `build_notebook.py` · This product uses the TMDB API but is not endorsed or certified by TMDB.*
